# 1. Dependencies

## System and OS

In [1]:
import os
import sys

## CUDA

`use_cuda` decides, in one place, whether this notebook runs on the GPU. Every model below reads it,
so turning it off is the whole of the CPU fallback.

`cuml.accel` patches scikit-learn's estimators to run on the card where it can. The three models that
need more than the patch provides — SVM, Random Forest and KNN — are imported from `cuml` directly
further down, so that what gets fitted, pickled and reloaded is never in doubt.

In [2]:
use_cuda = True

In [3]:
if use_cuda:
    cuda_root = os.path.join(sys.prefix, "targets", "x86_64-linux")
    if os.path.isdir(os.path.join(cuda_root, "include")):
        os.environ.setdefault("CUDA_PATH", cuda_root)

    %load_ext cuml.accel

cuML: Accelerator installed.


## Standard Library

In [97]:
import gc
import itertools
import math
import time
from typing import Any
from contextlib import contextmanager

## Serialization

In [5]:
import json

## Paralel

In [6]:
import joblib
from joblib import Parallel, delayed, parallel_config

### Utilities

In [98]:
def run_tasks_in_parallel(tasks) -> Any:
    """Run independent `delayed(...)` tasks in worker processes; results keep the order of the tasks.

    `inner_max_num_threads=1` stops pandas and pyarrow from starting their own thread pools inside
    each worker; without it, N processes x M threads would fight over the same cores and run slower.
    """
    with parallel_config(backend="loky", inner_max_num_threads=1,verbose=0):
        return Parallel(n_jobs=-1, verbose=0)(tasks)

## Data Processor

In [8]:
import numpy as np
import pandas as pd
import polars as pl

### Utilities

In [9]:
def get_all_file_names_in_folder(folder:str,file_extension:str):
    files = []
    for file in os.listdir(folder):
        if len(folder) > 0:
            if file.endswith(f".{file_extension}"):
                files.append(file)
    return sorted(files)


In [10]:
def filter_file_names(file_names: list[str], filter_word: str) -> list[str]:
    """Keep only the file names containing `filter_word`, e.g. a date like `"2018-02-14"`."""
    matching_file_names = []
    for file_name in file_names:
        if filter_word in file_name:
            matching_file_names.append(file_name)
    return matching_file_names

In [11]:
def get_split_folder(folder_name: str, split_name: str) -> str:
    """`data-split-label` + `dev` -> `data-split-label/dev`."""
    return os.path.join(folder_name, split_name)

In [12]:
def create_split_folders(folder_name: str, split_names: tuple[str, ...]) -> None:
    """Create one folder per split under `folder_name` if they do not exist yet."""
    for split_name in split_names:
        os.makedirs(get_split_folder(folder_name, split_name), exist_ok=True)

In [13]:
def get_split_file_paths(folder_name: str, split_name: str) -> list[str]:
    """Full paths of every Parquet file in one split, sorted the way the folder listing sorts them."""
    split_folder = get_split_folder(folder_name, split_name)
    return [
        os.path.join(split_folder, file_name)
        for file_name in get_all_file_names_in_folder(split_folder, "parquet")
    ]

In [14]:
def read_parquet_file(source_folder_name: str, file_name: str) -> pd.DataFrame:
    """Read one Parquet file from a folder into a DataFrame."""
    file_path = os.path.join(source_folder_name, file_name)
    return pd.read_parquet(file_path, engine="pyarrow")

In [15]:
def write_dataframe_to_parquet(
    df: pd.DataFrame, target_folder_name: str, file_name: str
) -> str:
    """Write one cleaned chunk and return the path it was written to."""
    output_path = os.path.join(target_folder_name, file_name)
    df.to_parquet(output_path, engine="pyarrow", compression="snappy", index=False)
    return output_path

In [16]:
def process_all_splits(
    source_folder_name: str,
    target_folder_name: str,
    split_names: tuple[str, ...],
    process_file,
    *arguments,
) -> None:
    """Run `process_file(source_split_folder, target_split_folder, file_name, *arguments)` over every
    file of every split, one worker process per file.

    The three steps that rewrite a whole split tree — imputation, label encoding and scaling — differ
    only in that function and in what it is handed, so the walking of the folders lives here.
    """
    create_split_folders(target_folder_name, split_names)

    for split_name in split_names:
        source_split_folder = get_split_folder(source_folder_name, split_name)
        target_split_folder = get_split_folder(target_folder_name, split_name)
        file_names = get_all_file_names_in_folder(source_split_folder, "parquet")

        run_tasks_in_parallel(
            delayed(process_file)(
                source_split_folder, target_split_folder, file_name, *arguments
            )
            for file_name in file_names
        )

        print(f"{split_name}: {len(file_names)} files written to {target_split_folder}/.")

In [17]:
def measure_feature_statistics(folder_name: str, split_name: str) -> pd.DataFrame:
    """Mean and standard deviation of every column of one split, measured from the files."""
    file_paths = get_split_file_paths(folder_name, split_name)

    frame = pl.scan_parquet(file_paths)
    column_names = frame.collect_schema().names()
    measured = frame.select(
        pl.col(column_names).mean().name.prefix("mean_"),
        pl.col(column_names).std().name.prefix("std_"),
    ).collect(engine="streaming").row(0, named=True)

    return pd.DataFrame(
        {
            "mean": {name: measured[f"mean_{name}"] for name in column_names},
            "std": {name: measured[f"std_{name}"] for name in column_names},
        }
    )

In [18]:
def load_label_classes(input_file: str) -> list[str]:
    """Read the stored class list; a name's position in the list is its class id."""
    with open(input_file) as file:
        return json.load(file)

In [19]:
def count_labels_in_file(folder_name: str, file_name: str, label_column: str) -> pd.Series:
    """Count the rows per class in one label file."""
    labels = read_parquet_file(folder_name, file_name)
    return labels[label_column].value_counts()

In [20]:
def count_labels_in_folder(folder_name: str, label_column: str) -> pd.Series:
    """Count the rows per class across every label file in one folder."""
    file_names = get_all_file_names_in_folder(folder_name, "parquet")

    counts_per_file = run_tasks_in_parallel(
        delayed(count_labels_in_file)(folder_name, file_name, label_column)
        for file_name in file_names
    )

    return pd.concat(counts_per_file).groupby(level=0).sum()

## Pre-Processing

In [34]:
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import IncrementalPCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

## Plotting

In [35]:
import matplotlib
import matplotlib.pyplot as plt

## Models

Each import picks the GPU implementation when `use_cuda` is set and the scikit-learn one otherwise.
The two exceptions are explained where they are: Keras builds the softmax regression on either device
with the same code, and XGBoost ships one class for both and is told which device to use by argument.

### K-Nearest Neighbours

In [36]:
if use_cuda:
    from cuml.neighbors import KNeighborsClassifier, NearestNeighbors
else:
    from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

### Support Vector Machine

In [37]:
if use_cuda:
    from cuml.svm import SVC
else:
    from sklearn.svm import SVC

### Random Forest

In [38]:
if use_cuda:
    from cuml.ensemble import RandomForestClassifier
else:
    from sklearn.ensemble import RandomForestClassifier

### Logistic Regression (Softmax)

TensorFlow claims the whole card on its first allocation, which would leave nothing for the cuML
models above. Memory growth has to be enabled before any GPU is initialised, which is why it belongs
here with the import rather than beside the model.

In [39]:
import tensorflow as tf
from tensorflow import keras

if use_cuda:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    tf.config.set_visible_devices([], "GPU")

I0000 00:00:1789741176.791182    2172 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789741176.832870    2172 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789741177.642362    2172 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1789741177.947619    2742 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that

### XGBoost

One class for both backends — the card is chosen with the `device` argument rather than by importing
from somewhere else, so there is no `use_cuda` fork here. That argument needs `xgboost >= 2.0`, and
`cuml.accel` does not patch XGBoost, so this is the upstream estimator rather than a proxy over it.

In [40]:
from xgboost import XGBClassifier

## Evaluation

In [41]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)

### Utilities

Everything the five model sections share. Each one is here rather than in a model section because all
five need it: the conversions every estimator's input has to go through, the subsampler the searches
rely on, the grid runner, the chunked prediction loop, and the four metrics every report is built
from.

In [64]:
def to_features(data) -> np.ndarray:
    """Anything tabular -> the C-contiguous `float32` matrix cuML and XGBoost expect."""
    if isinstance(data, (pl.DataFrame, pl.Series)):
        data = data.to_numpy()
    return np.ascontiguousarray(data, dtype=np.float32)

In [65]:
def to_labels(data) -> np.ndarray:
    """Anything one-dimensional -> `int32` class ids; cuML classifiers reject float labels."""
    if isinstance(data, (pl.DataFrame, pl.Series)):
        data = data.to_numpy()
    return np.asarray(data).ravel().astype(np.int32)

In [66]:
def to_numpy(data) -> np.ndarray:
    """A prediction from any backend -> a flat NumPy vector (`.get()` pulls a cupy array back)."""
    if hasattr(data, "to_numpy"):
        return np.asarray(data.to_numpy()).ravel()
    if hasattr(data, "get"):
        return data.get().ravel()
    return np.asarray(data).ravel()

In [67]:
def to_numpy_2d(data) -> np.ndarray:
    """The same for a two-dimensional result.

    `to_numpy` flattens, which is right for a prediction vector and wrong for the `(n_query, k)`
    shape a `kneighbors` call returns.
    """
    if hasattr(data, "to_numpy"):
        return np.asarray(data.to_numpy())
    if hasattr(data, "get"):
        return np.asarray(data.get())
    return np.asarray(data)

In [68]:
def free_gpu_memory() -> None:
    """Hand back the device memory a finished model was holding.

    cuML allocates from RMM/CuPy pools that only release once the Python objects are collected, so
    `del model` alone is not enough — over a grid the card grows by roughly the size of a model per
    fit and never comes down. Collecting and then draining the pools returns it between combinations.
    """
    gc.collect()
    if not use_cuda:
        return
    try:
        import cupy as cp

        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass

In [69]:
@contextmanager
def step(message: str):
    """Announce a blocking stage, then tick it off when it returns.

    `SVC.fit` is one opaque call with nothing inside it to count. Printing before entering and
    completing the line afterwards says which stage is running and which have finished — no timer and
    no background thread, just the two things that are actually known.
    """
    print(f"  {message} ...", end="", flush=True)
    try:
        yield
    except BaseException:
        print(" failed")
        raise
    print(" done")

In [70]:
def progress_bar(done: int, total: int, label: str = "", width: int = 30) -> None:
    """Redraw a one-line bar, driven by the caller's own loop so no thread is needed."""
    filled = width if total <= 0 else int(width * done / total)
    line = f"  [{'#' * filled}{'-' * (width - filled)}] {done:,}/{total:,}"
    if label:
        line += f" {label}"
    print(line.ljust(100), end="\n" if done >= total else "\r", flush=True)

In [71]:
def stratified_subsample(
    x, y, n_samples: int | None, random_state: int, min_per_class: int = 1
) -> tuple[np.ndarray, np.ndarray]:
    """Take a class-proportional sample of the rows, as `float32` features and `int32` labels.

    `n_samples=None`, or a budget at least as large as the split, returns everything unchanged — so
    the same call works for a search sample and for a final fit on the whole split.

    `min_per_class` is a floor under the proportional quota. Without it a class holding 3 of 100,000
    rows contributes a single row, and macro F1 — the criterion the searches select on — then swings
    by a full 1/15 on whether that one row happens to be classified correctly.
    """
    labels = to_labels(y)
    total = labels.shape[0]

    if n_samples is None or n_samples >= total:
        return to_features(x), labels

    generator = np.random.default_rng(random_state)
    classes, counts = np.unique(labels, return_counts=True)

    quota = np.floor(counts / total * n_samples).astype(np.int64)
    quota = np.maximum(quota, np.minimum(min_per_class, counts))
    quota = np.minimum(quota, counts)

    selected = []
    for class_id, class_quota in zip(classes, quota):
        class_rows = np.flatnonzero(labels == class_id)
        if class_quota < class_rows.size:
            class_rows = generator.choice(class_rows, size=int(class_quota), replace=False)
        selected.append(class_rows)

    keep = np.zeros(total, dtype=bool)
    keep[np.sort(np.concatenate(selected))] = True
    subset = x.filter(pl.Series(keep)) if isinstance(x, pl.DataFrame) else x[keep]

    print(f"  subsampled {total:,} -> {int(keep.sum()):,} rows across {classes.size} classes")
    return to_features(subset), labels[keep]

In [72]:
def build_grid(*value_lists) -> list[tuple]:
    """Every combination of the given value lists, in a fixed, repeatable order."""
    return [tuple(combination) for combination in itertools.product(*value_lists)]

In [73]:
def run_search(
    combinations: list[tuple],
    parameter_names: tuple[str, ...],
    fit_and_score,
    search_data: tuple,
    score_column: str,
) -> pd.DataFrame:
    """Fit and score every combination of a grid, and keep going when one of them fails.

    `fit_and_score(*combination, *search_data)` returns a dict of everything measured for that
    combination — its scores and whatever it cost. A combination that raises is recorded with its
    error text and no scores, rather than ending the search: on the GPU the failures are specific
    combinations (a solver refusing a parameter, a forest too large for the card), and the remaining
    ones are still worth having.
    """
    rows = []
    total = len(combinations)

    for number, combination in enumerate(combinations, start=1):
        parameters = dict(zip(parameter_names, combination))
        described = " ".join(f"{name}={value}" for name, value in parameters.items())
        print(f"[{number}/{total}] {described}")

        try:
            measurements = fit_and_score(*combination, *search_data)
        except Exception as error:
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"  !! failed, skipped: {reason}")
            free_gpu_memory()
            rows.append({**parameters, "error": reason})
            continue

        rows.append({**parameters, **measurements})
        print(f"  {score_column}={measurements[score_column]:.4f}")

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Scored {total - failed} of {total} combinations" + (f", {failed} failed." if failed else "."))

    if score_column not in results:
        return results
    return results.sort_values(score_column, ascending=False, ignore_index=True)

In [74]:
def save_search_results(results: pd.DataFrame, file_name: str, folder_name: str) -> str:
    """Write a search table to CSV, so a grid that took hours is not held only in memory."""
    os.makedirs(folder_name, exist_ok=True)
    file_path = os.path.join(folder_name, file_name)
    results.to_csv(file_path, index=False)
    print(f"Saved search results to {file_path}.")
    return file_path

In [75]:
def get_best_hyperparameter(results: pd.DataFrame, score_column: str) -> dict:
    """Return the top row of a search table as a dict, and print what won."""
    if score_column not in results:
        raise ValueError(
            f"None of the {len(results)} combinations in this grid was scored — every one of them "
            "failed, so there is nothing to choose between. The `error` column says why."
        )

    best = results.sort_values(score_column, ascending=False).iloc[0]
    print(f"Best by {score_column}:")
    for name, value in best.items():
        print(f"  {name:>18} = {value}")
    return best.to_dict()


In [76]:
def dump_trained_model(model, name: str, subfolder: str, folder_name: str) -> str:
    """Pickle a fitted model under `folder_name/subfolder/name`; return where it went."""
    target_folder = os.path.join(folder_name, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    joblib.dump(model, file_path)
    return file_path

In [77]:
def load_trained_model(name: str, subfolder: str, folder_name: str):
    """Read back a model `dump_trained_model` wrote."""
    return joblib.load(os.path.join(folder_name, subfolder, name))

In [78]:
def save_array_atomically(array: np.ndarray, file_path: str) -> None:
    """Write to a temporary file and rename it into place.

    A prediction run that is interrupted mid-write would otherwise leave a truncated `.npy` that the
    next run reads as a finished chunk. A rename is atomic, so a file either exists complete or not
    at all.
    """
    temporary_path = file_path + ".tmp"
    with open(temporary_path, "wb") as file:
        np.save(file, array)
    os.replace(temporary_path, file_path)

In [79]:
def predict_in_chunks(
    predict_chunk, x, cache_folder: str, prefix: str, chunk_size: int
) -> np.ndarray:
    """Predict a whole split a chunk at a time, keeping each chunk on disk.

    `predict_chunk(rows)` is the only model-specific part: it takes the rows of one chunk and returns
    their class ids. Everything else — the slicing, the caching, the bar — is the same for all five
    models.

    The chunks are cached because predicting 3.2 million rows takes minutes to hours depending on the
    model, and an interrupted run should resume rather than restart. A chunk that is already on disk
    is not recomputed, which also means **stale chunks are silently reused**: delete the folder after
    retraining a model under the same name.
    """
    os.makedirs(cache_folder, exist_ok=True)
    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows).")
        return np.empty(0, dtype=np.int32)

    n_chunks = math.ceil(total / chunk_size)
    print(f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,} into {cache_folder}/")

    file_paths, computed, cached = [], 0, 0
    for number, start in enumerate(range(0, total, chunk_size), start=1):
        file_path = os.path.join(cache_folder, f"{prefix}_{number:05d}.npy")
        file_paths.append(file_path)

        if os.path.exists(file_path):
            cached += 1
        else:
            rows = (
                x.slice(start, chunk_size)
                if isinstance(x, (pl.DataFrame, pl.Series))
                else x[start : start + chunk_size]
            )
            save_array_atomically(np.asarray(predict_chunk(rows), dtype=np.int32), file_path)
            computed += 1

        progress_bar(number, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    return np.concatenate([np.load(file_path) for file_path in file_paths])

In [80]:
def load_prediction_chunks(cache_folder: str) -> np.ndarray:
    """Read back every cached prediction chunk of one split, in file order."""
    if not os.path.isdir(cache_folder):
        raise FileNotFoundError(cache_folder)

    file_names = get_all_file_names_in_folder(cache_folder, "npy")
    if not file_names:
        raise FileNotFoundError(f"No prediction chunks in {cache_folder}/.")

    return np.concatenate(
        [np.load(os.path.join(cache_folder, file_name)) for file_name in file_names]
    )

In [81]:
def get_present_labels(true, pred) -> list[int]:
    """The class ids that appear in either the truth or the prediction, in order."""
    return sorted({int(x) for x in np.unique(true)} | {int(x) for x in np.unique(pred)})

In [82]:
def evaluate(pred, true, labels: list[int] | None = None) -> pd.DataFrame:
    """Accuracy and macro precision/recall/F1 in one row.

    The macro averages weight every class equally, so the 18 `SQL Injection` rows of the dev split
    count as much as its 649,000 `Benign` rows. That is the point: accuracy on a split that is 83%
    one class says almost nothing about the other fourteen.
    """
    if labels is None:
        labels = get_present_labels(true, pred)

    return pd.DataFrame(
        [
            {
                "accuracy": accuracy_score(true, pred),
                "precision_macro": precision_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "recall_macro": recall_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "f1_macro": f1_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
            }
        ]
    )

In [83]:
def get_classification_report(pred, true, class_names: list[str]) -> pd.DataFrame:
    """Precision, recall, F1 and support for every class, by name."""
    labels = get_present_labels(true, pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        true, pred, labels=labels, average=None, zero_division=0
    )
    return pd.DataFrame(
        {
            "class": [class_names[label] for label in labels],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )

In [84]:
def get_confusion_matrix(pred, true, class_names: list[str]) -> pd.DataFrame:
    """Rows are what the traffic was, columns are what the model called it."""
    labels = get_present_labels(true, pred)
    present_names = [class_names[label] for label in labels]
    return pd.DataFrame(
        confusion_matrix(true, pred, labels=labels), index=present_names, columns=present_names
    )

In [85]:
def get_evaluation_results(cache_folder: str, true_labels) -> pd.DataFrame:
    """Score the cached predictions of one split against its labels."""
    pred = load_prediction_chunks(cache_folder)
    true = to_labels(true_labels)

    if len(pred) != len(true):
        raise ValueError(
            f"{len(pred):,} predictions against {len(true):,} labels in {cache_folder}/ — "
            "the cache belongs to a different split, or was written by a different run."
        )
    return evaluate(pred=pred, true=true)

# 2. Reformatting

## 2.1. Change CSV to Parquet

The raw CSE-CIC-IDS2018 dataset ships as 10 daily CSV files (~6.7 GB). CSV is slow to read, stores
every value as text and carries no schema, so the first pre-processing step rewrites it as Parquet:
columnar, compressed, typed, and readable one piece at a time.

**In:** `cse-cic-ids2018/*.csv` $\rightarrow$ **Out:** `data-raw/*.parquet` (one file per chunk of rows)

Three things have to be repaired on the way, because the dataset is not uniform:

| Problem | Where it happens |
| --- | --- |
| `2018-02-20` carries 4 extra identifier columns (`Flow ID`, `Src IP`, `Src Port`, `Dst IP`) | 1 of 10 files | 
| The CSV header is repeated *inside* the file as a data row | `2018-02-16` (1×), `2018-02-28` (33×), `2018-03-01` (25×) |
| Numbers are read as text whenever a column contains one of those header rows | the files above |

The section is built bottom-up, so each part can be read on its own:

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 2.1.1 Settings | Where are the files, how big is a chunk? | — |
| 2.1.2 Column schema | Which columns does every output file have? | 2.1.1 |
| 2.1.3 Cleaning one chunk | How is a batch of rows made trustworthy? | 2.1.1, 2.1.2 |
| 2.1.4 Writing Parquet | What is a file called and how is it written? | 2.1.1 |
| 2.1.5 Converting one CSV | How is a single day converted? | 2.1.3, 2.1.4 |
| 2.1.6 Converting the folder | How are all 10 days run in parallel? | 2.1.2, 2.1.5 |
| 2.1.7 Run | — | 2.1.1, 2.1.6 |
| 2.1.8 Check the result | Did it write what we expect? | 2.1.1 |


### 2.1.1. Settings

Everything this step has to be *told*, kept in one place. The functions below take these as
arguments (with the constants as defaults), so any of them can also be called on a different
folder or chunk size without editing the function.

In [22]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [23]:
CHUNK_SIZE = 100_000

In [24]:
LABEL_COLUMN = "label"
FEATURE_DTYPE = "float32"

In [25]:
COLUMNS_TO_DROP = {
    "flow id",
    "src ip",
    "source ip",
    "src port",
    "source port",
    "dst ip",
    "destination ip",
    "timestamp",
}

### 2.1.2. Column Schema

*Which columns should every Parquet file have?*

Decided once, from the header rows alone — no data is read here. The schema is the union of the
column names of all CSVs, in first-seen order, minus `COLUMNS_TO_DROP`. Because `2018-02-20` is the
only file that differs and all four of its extra columns are dropped, the result is the same 79
columns for every day, which is what lets later steps read the whole folder as one table.

This is the **only** place `COLUMNS_TO_DROP` is used: one decision, in one place. 2.1.3 then forces
each chunk of data into the layout decided here.

In [26]:
def normalize_column_names(column_names) -> list[str]:
    """Trim surrounding spaces and lowercase, so `" Dst Port"` and `"dst port"` are one name."""
    columns = []
    for column in column_names:
        columns.append(str(column).strip().lower())
    return columns

In [27]:
def exclude_unwanted_columns(
    column_names: list[str], columns_to_drop: set[str] = COLUMNS_TO_DROP
) -> list[str]:
    """Keep only the column names worth storing, preserving their order."""
    remaining_columns = []
    for column in column_names:
        if column not in columns_to_drop:
            remaining_columns.append(column)
    return remaining_columns

In [28]:
def read_csv_column_names(source_folder_name: str, file_name: str) -> list[str]:
    """Read the header row of one CSV — `nrows=0` loads no data — and normalize its names."""
    file_path = os.path.join(source_folder_name, file_name)
    header_only = pd.read_csv(file_path, nrows=0)
    return normalize_column_names(header_only.columns)

In [29]:
def create_column_schema(
    source_folder_name: str, columns_to_drop: set[str] = COLUMNS_TO_DROP
) -> list[str]:
    """Build the column layout shared by every Parquet file written from this folder."""
    column_schema: list[str] = []
    for file_name in get_all_file_names_in_folder(source_folder_name, "csv"):
        file_columns = read_csv_column_names(source_folder_name, file_name)
        for column_name in exclude_unwanted_columns(file_columns, columns_to_drop):
            if column_name not in column_schema:
                column_schema.append(column_name)
    return column_schema

In [30]:
column_schema = create_column_schema(PATH_FOLDER_CSV)
print(f"{len(column_schema)} columns: {column_schema[:3]} ... {column_schema[-2:]}")

79 columns: ['dst port', 'protocol', 'flow duration'] ... ['idle min', 'label']


### 2.1.3. Cleaning One Chunk

*How is a batch of raw rows made trustworthy?*

Four small transformations, each doing one thing, and `clean_chunk` running them in order. They all
take a `DataFrame` and return one, so the order is visible at a glance and any single step can be
tried on its own chunk while reading.

1. `standardize_column_names` — same naming rule as the schema, so the two can be matched.
2. `align_to_column_schema` — same columns, same order, in every file. This is also what removes
   the dropped identifier columns: they are simply not in the schema.
3. `cast_features_to_numeric` — text to float; anything unparseable becomes `NaN`.
4. `drop_invalid_label_rows` — throws away repeated header rows and rows without a label.

Order matters: the columns must be renamed before they can be matched against the schema, and the
label must still be text when the repeated header rows are detected — which is why the label is the
one column step 3 leaves alone.

In [31]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Rename the columns of a chunk with the same rule `create_column_schema` used."""
    df.columns = normalize_column_names(df.columns)
    return df

In [32]:
def align_to_column_schema(df: pd.DataFrame, column_schema: list[str]) -> pd.DataFrame:
    """Force a chunk into the shared layout: schema columns only, in schema order.

    Columns the schema does not list (the dropped identifiers) disappear, and columns this file
    does not have would come back filled with `NaN`, so every output file has the same shape.
    """
    return df.reindex(columns=column_schema)

In [ ]:
def cast_features_to_numeric(
    df: pd.DataFrame, label_column: str = LABEL_COLUMN, datatype: Any = FEATURE_DTYPE
) -> pd.DataFrame:
    """Convert every column except the label to a float.

    A column that contains a repeated header row is read as text by pandas, which would make the
    whole column text. `errors="coerce"` turns those values — and any blank — into `NaN`; the rows
    they came from are dropped next, and genuinely missing values are filled in 2.5 (Imputation).
    """
    feature_columns = [name for name in df.columns if name != label_column]
    df[feature_columns] = df[feature_columns].apply(pd.to_numeric, errors="coerce").astype(datatype)
    return df

In [86]:
def drop_invalid_label_rows(df: pd.DataFrame, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Remove rows that are not observations.

    Some files repeat their CSV header in the middle of the data (2018-02-28 does it 33 times),
    which pandas reads as an ordinary row whose label is the literal text `"Label"`. Those rows go,
    together with any row that has no label at all.
    """
    label_values = df[label_column].astype("string").str.strip().str.lower()
    is_repeated_header = label_values == label_column
    return df[~is_repeated_header].dropna(subset=[label_column])

In [ ]:
def clean_chunk(df: pd.DataFrame, column_schema: list[str]) -> pd.DataFrame:
    """Run one chunk of raw CSV rows through the four steps above, in order."""
    df = standardize_column_names(df)
    df = align_to_column_schema(df, column_schema)
    df = cast_features_to_numeric(df)
    df = drop_invalid_label_rows(df)
    return df

### 2.1.4. Writing Parquet Files

*What is an output file called, and how is it written?*

One chunk becomes one file. Chunk numbers are zero-padded so the files of a day sort in the order
they were read, and the source CSV name is kept as the prefix — the train/validation/test split in
2.4 selects files by the date in that prefix, so the naming is not cosmetic.

`snappy` compression is the fast-to-decompress default; these files are read many times in the
steps that follow, so read speed matters more than the last few percent of disk space.

In [89]:
def build_parquet_file_name(csv_file_name: str, chunk_number: int) -> str:
    """`2018-02-14-Wednesday_....csv` + chunk 7 -> `2018-02-14-Wednesday_..._00007.parquet`."""
    base_name = os.path.splitext(csv_file_name)[0]
    return f"{base_name}_{chunk_number:05d}.parquet"

### 2.1.5. Converting One CSV File

*How is a single day converted?*

The two functions that turn 2.1.3 and 2.1.4 into a stream: read a fixed number of rows, clean them,
write them, forget them. Peak memory is one chunk, not one file, so the 4 GB `2018-02-20` costs no
more than the 108 MB `2018-03-01`.

`low_memory=False` makes pandas look at the whole column before choosing its type, instead of
guessing per block and reporting mixed types for the columns that contain a repeated header row.

In [90]:
def read_csv_in_chunks(source_folder_name: str, file_name: str, chunk_size: int = CHUNK_SIZE):
    """Iterate over one CSV `chunk_size` rows at a time."""
    file_path = os.path.join(source_folder_name, file_name)
    return pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)

In [91]:
def convert_csv_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    column_schema: list[str],
    chunk_size: int = CHUNK_SIZE,
) -> int:
    """Convert one CSV into a numbered series of Parquet files; return how many were written."""
    written_files = 0
    for chunk_number, chunk in enumerate(
        read_csv_in_chunks(source_folder_name, file_name, chunk_size), start=1
    ):
        cleaned_chunk = clean_chunk(chunk, column_schema)
        write_dataframe_to_parquet(
            cleaned_chunk,
            target_folder_name,
            build_parquet_file_name(file_name, chunk_number),
        )
        written_files += 1
    return written_files

### 2.1.6. Converting Every CSV File

*How is the whole folder run?*

One task per CSV file, handed to `run_tasks_in_parallel` (section 1). Each worker process takes one
whole CSV, so the workers never write to the same file and no result has to be sent back except a
count.

The schema is built **once, before** the workers start, and passed in — if every worker derived its
own, a file would be aligned to a layout the others do not share.

In [99]:
def convert_all_csv_files_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE,
) -> None:
    """Convert every CSV in the source folder to Parquet, one worker process per file."""
    csv_file_names = get_all_file_names_in_folder(source_folder_name, "csv")
    column_schema = create_column_schema(source_folder_name)
    os.makedirs(target_folder_name, exist_ok=True)

    written_per_file: list[int] = run_tasks_in_parallel(
        delayed(convert_csv_file_to_parquet)(
            source_folder_name,
            target_folder_name,
            file_name,
            column_schema,
            chunk_size,
        )
        for file_name in csv_file_names
    )

    print(
        f"Converted {len(csv_file_names)} CSV files "
        f"into {sum(written_per_file)} Parquet files in {target_folder_name}/."
    )

### 2.1.7. Run the Conversion

Reads `cse-cic-ids2018/` and fills `data-raw/`. Roughly 6.7 GB in, and it is the slowest step in the
notebook — everything after this reads Parquet, so it only has to be run once.

In [ ]:
convert_all_csv_files_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW)

# 3. Exploratory Data Analysis (EDA)

# 4. Transform Infinite Values to NaN

Two of the 78 feature columns are rates that CICFlowMeter computes as a total divided by the duration
of the flow: `flow byts/s` and `flow pkts/s`. A flow that starts and ends inside the same microsecond
has a duration of `0`, and the division then returns $\infty$ (or $-\infty$). Section 2.1 stores such
a value exactly as the CSV reported it, so `data-raw/` carries **131,799** of them.

$\infty$ is not something the later steps can work with. The mean or the standard deviation of a
column holding a single infinity is itself $\infty$ or `NaN`, so one impossible cell would spread to
its whole column the moment the data is imputed, scaled or projected.

`NaN` is a value the pipeline already handles: the imputation step later in this notebook replaces
every `NaN` with the median of its column. Rewriting $\infty$ as `NaN` therefore does not throw the
row away — it routes an impossible rate through the same repair as a missing one.

**In:** `data-raw/*.parquet` $\rightarrow$ **Out:** `data-no-inf/*.parquet`

Same file names, same rows, same columns; only the infinite cells change. The names are kept because
the train/validation/test split picks files by the date in the file name.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 4.1 Settings | Where do the files go? | — |
| 4.2 Replacing in one file | What is actually changed? | — |
| 4.3 Replacing in every file | How are all 168 files run? | 4.2 |
| 4.4 Run | — | 4.1, 4.3 |
| 4.5 Check the result | Did the infinities really go? | 4.1 |

## 4.1. Settings

Only the destination is new. The input is `PATH_FOLDER_RAW` from 2.1.1 — this step reads exactly what
the conversion wrote. Both folders are passed as arguments in 4.4 instead of being baked into the
functions, so the same code can clean any folder of Parquet files.

In [ ]:
PATH_FOLDER_NO_INF = "data-no-inf"

## 4.2. Replacing the Infinite Values in One File

*What is actually changed?*

Two functions, split by what each has to know. `replace_infinite_values` knows only about a
`DataFrame`, so it can be tried on a handful of rows while reading; `replace_infinite_values_in_file`
knows only about files, and is the unit of work a worker process runs in 4.3.

Only the numeric columns are touched. `label` is text, cannot hold an infinity, and is passed through
untouched — this step can never change a class. It is also why the counting function in 4.5 looks at
the same numeric columns: what is replaced and what is counted have to be the same set.

In [ ]:
def replace_infinite_values(df: pd.DataFrame) -> pd.DataFrame:
    """Turn `+inf` and `-inf` into `NaN` in every numeric column, leaving the label alone."""
    numeric_columns = df.select_dtypes(include=np.number).columns
    df[numeric_columns] = df[numeric_columns].replace([np.inf, -np.inf], np.nan)
    return df

In [ ]:
def replace_infinite_values_in_file(
    source_folder_name: str, target_folder_name: str, file_name: str
) -> str:
    """Clean one Parquet file into the target folder under the same name; return the path written."""
    df = read_parquet_file(source_folder_name, file_name)
    df = replace_infinite_values(df)
    return write_dataframe_to_parquet(df, target_folder_name, file_name)

## 4.3. Replacing Them in Every File

*How are all 168 files run?*

The same shape as 2.1.6, one task per Parquet file instead of one per CSV: the files are independent,
every worker reads one and writes one, and the only thing sent back is the path it wrote. The target
folder is created before the workers start, so none of them race to create it.

In [ ]:
def replace_infinite_values_in_folder(source_folder_name: str, target_folder_name: str) -> None:
    """Rewrite every Parquet file in the source folder with its infinite values turned into `NaN`."""
    file_names = get_all_file_names_in_folder(source_folder_name, "parquet")
    os.makedirs(target_folder_name, exist_ok=True)

    written_files = run_tasks_in_parallel(
        delayed(replace_infinite_values_in_file)(source_folder_name, target_folder_name, file_name)
        for file_name in file_names
    )

    print(f"Rewrote {len(written_files)} Parquet files into {target_folder_name}/.")

## 4.4. Run the Replacement

Reads `data-raw/` and fills `data-no-inf/`: 168 files, roughly 1.9 GB, identical except for the
infinite cells. Much cheaper than 2.1.7 — the columns are already typed, so this is one pass of
reading and writing Parquet.

In [ ]:
replace_infinite_values_in_folder(PATH_FOLDER_RAW, PATH_FOLDER_NO_INF)

## 4.5. Check the Result

*Did the infinities really go?*

One counting function, run on the folder before and the folder after, is what turns the claim into
evidence. It counts **per column** rather than reporting one total, because *where* the infinities
sit is the part that explains them: a count concentrated in the two rate columns is the division by a
zero duration described above, and shows that no other column has quietly acquired one.

`count_infinite_values_in_file` is the mirror of `replace_infinite_values` — same numeric columns,
opposite question. Adding up the per-file counts gives the per-column count of the whole folder, and
only the columns that still have one are returned, so a clean folder returns an empty result.

Both cells below read a full folder, so each takes about as long as 4.4 itself.

In [ ]:
def count_infinite_values_in_file(source_folder_name: str, file_name: str) -> pd.Series:
    """Count the infinite values per numeric column in one Parquet file."""
    df = read_parquet_file(source_folder_name, file_name)
    numeric_columns = df.select_dtypes(include=np.number)
    return np.isinf(numeric_columns).sum()

In [ ]:
def count_infinite_values(source_folder_name: str) -> pd.Series:
    """Count the infinite values in a folder of Parquet files, per column."""
    file_names = get_all_file_names_in_folder(source_folder_name, "parquet")

    counts_per_file = run_tasks_in_parallel(
        delayed(count_infinite_values_in_file)(source_folder_name, file_name)
        for file_name in file_names
    )

    counts_per_column = sum(counts_per_file)
    print(
        f"{int(counts_per_column.sum()):,} infinite values "
        f"in {len(file_names)} Parquet files in {source_folder_name}/."
    )
    return counts_per_column[counts_per_column > 0]

**Before the replacement:**

In [ ]:
count_infinite_values(PATH_FOLDER_RAW)

**After the replacement:**

In [ ]:
count_infinite_values(PATH_FOLDER_NO_INF)

`data-raw/` holds 131,799 infinite values, and all of them are in the two rate columns —
`flow pkts/s` (95,760) and `flow byts/s` (36,039). The other 76 feature columns have none, which is
what the division by a zero duration predicts. The `float32` cast of 2.1.3 adds none of its own
either: the largest finite magnitude in the dataset is about $9.8 \times 10^{11}$, far below the
`float32` ceiling of about $3.4 \times 10^{38}$.

`data-no-inf/` returns an empty result: not a single column has an infinite value left. The rows that
carried them are still there, now holding `NaN` in those two columns, for the imputation step to fill
with the column median.

# 5. Split Datasets

The three subsets are cut here, before anything is *learned* from the data. Every step after this one
— the imputation medians, the scaler, the PCA components, SMOTE — is fitted on the training split
alone, so the training split has to exist first. Fitting them on all the rows and splitting afterwards
would leak the validation and test rows into the model's preparation and quietly flatter every result
that follows.

**In:** `data-no-inf/*.parquet` $\rightarrow$ **Out:** `data-split-feature/{train,dev,test}/*.parquet`
and `data-split-label/{train,dev,test}/*.parquet`

### Why the split is stratified by label, not by day

CSE-CIC-IDS2018 was captured one attack scenario at a time, so a class and a date are nearly the same
thing. Counted from `data-no-inf/`:

| Date | Classes in that day's files (rows) |
| --- | --- |
| 2018-02-14 | Benign (667,626), FTP-BruteForce (193,360), SSH-Bruteforce (187,589) |
| 2018-02-15 | Benign (996,077), DoS attacks-GoldenEye (41,508), DoS attacks-Slowloris (10,990) |
| 2018-02-16 | DoS attacks-Hulk (461,912), Benign (446,772), DoS attacks-SlowHTTPTest (139,890) |
| 2018-02-20 | Benign (7,372,557), DDoS attacks-LOIC-HTTP (576,191) |
| 2018-02-21 | DDOS attack-HOIC (686,012), Benign (360,833), DDOS attack-LOIC-UDP (1,730) |
| 2018-02-22 | Benign (1,048,213), Brute Force -Web (249), Brute Force -XSS (79), SQL Injection (34) |
| 2018-02-23 | Benign (1,048,009), Brute Force -Web (362), Brute Force -XSS (151), SQL Injection (53) |
| 2018-02-28 | Benign (544,200), Infilteration (68,871) |
| 2018-03-01 | Benign (238,037), Infilteration (93,063) |
| 2018-03-02 | Benign (762,384), Bot (286,191) |

Handing whole days to a subset — say six days to train and two each to validation and test — would
therefore hand whole *classes* to a subset. A model that never saw a Bot flow while training would be
tested on 2018-03-02. That measures generalisation to attacks the model has never met, which is a
different experiment from this one: this study measures robustness against disturbed input, and needs
every class present in all three subsets.

So the rows of every file are split **stratified by `label`**, 60/20/20. Each class keeps the same
proportion in all three subsets, down to the 87 SQL Injection flows in the entire dataset, which
become 52 / 18 / 17.

A day-based split is still worth running as a *separate* experiment. It is a different question, not
a different implementation of this one.

### Why every file is split on its own

16.2 million rows never have to be in memory at once. Each Parquet file holds at most 100,000 rows
(2.1.1) and is split by itself, which is not an approximation: 168 stratified 60/20/20 splits add up
to a stratified 60/20/20 split of the whole, as 5.8 measures.

It does have one requirement. The two-stage split below needs at least **4** rows of a class *within a
single file* — with fewer, the second stage is left with one row of that class and `train_test_split`
refuses to stratify. The smallest class inside any file of this dataset is 9 rows, so every one of the
168 files splits cleanly; a dataset with rarer classes would need the files concatenated per day
first.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 5.1 Settings | Where do the files go, in what proportion? | — |
| 5.2 Folder layout | Which folder does a split end up in? | 5.1 |
| 5.3 Splitting the rows | How are the rows of one file divided? | 5.1 |
| 5.4 Writing one split | Why are features and labels written apart? | 5.2 |
| 5.5 Splitting one file | How is one file handled start to finish? | 5.3, 5.4 |
| 5.6 Splitting one day | How is a day run? | 5.2, 5.5 |
| 5.7 Run | — | 5.1, 5.6 |
| 5.8 Check the result | Did every class survive in all three? | 5.1, 5.2 |

## 5.1. Settings

The input folder is `PATH_FOLDER_NO_INF` (4.1) and the label column is `LABEL_COLUMN` (2.1.1): both
describe the data this step is handed, not a decision it makes. What this step decides is the two
destinations, the proportions, the seed and which days exist.

`dev` is the folder name of the validation split — kept short because it is part of every path from
here on.

`TRAIN_SIZE` is derived rather than typed, so the three proportions can never disagree: the split
takes the two held-out shares and gives the remainder to training.

`RANDOM_STATE` is what makes the split reproducible. The same seed is used for every file and both
stages, so re-running this section rebuilds the exact same three subsets — which is what lets the
models trained later be compared to each other at all.

In [ ]:
PATH_FOLDER_SPLIT_FEATURE = "data-split-feature"
PATH_FOLDER_SPLIT_LABEL = "data-split-label"

In [ ]:
SPLIT_NAMES = ("train", "dev", "test")

In [ ]:
DEV_SIZE = 0.20
TEST_SIZE = 0.20
TRAIN_SIZE = 1.0 - DEV_SIZE - TEST_SIZE

In [ ]:
RANDOM_STATE = 42

In [ ]:
COLLECTION_DAYS = (
    "2018-02-14",  # FTP-BruteForce, SSH-Bruteforce
    "2018-02-15",  # DoS attacks-GoldenEye, DoS attacks-Slowloris
    "2018-02-16",  # DoS attacks-Hulk, DoS attacks-SlowHTTPTest
    "2018-02-20",  # DDoS attacks-LOIC-HTTP
    "2018-02-21",  # DDOS attack-HOIC, DDOS attack-LOIC-UDP
    "2018-02-22",  # Brute Force -Web, Brute Force -XSS, SQL Injection
    "2018-02-23",  # Brute Force -Web, Brute Force -XSS, SQL Injection
    "2018-02-28",  # Infilteration
    "2018-03-01",  # Infilteration
    "2018-03-02",  # Bot
)

## 5.2. Where Each File Goes

*Which folder does a split end up in?*

Six folders — three splits, with features and labels kept apart. The layout itself is decided by
`get_split_folder` in section 1 (`data-split-label` + `dev` $\rightarrow$ `data-split-label/dev`),
because every later step addresses the same `folder/split` shape: 5.4 and 5.8 use it here, and
section 6 reads `data-split-feature/train` and writes `data-imputed/train` through it.

What belongs to *this* section is that there are two trees rather than one, so the only function left
here is the one that makes the split folders under both.

They are created once, before any worker starts (5.6). `exist_ok=True` is what lets the ten days
write into the same six folders and makes a re-run harmless.

In [ ]:
def create_feature_and_label_folders(
    features_folder_name: str, labels_folder_name: str, split_names: tuple[str, ...] = SPLIT_NAMES
) -> None:
    """Create the three feature folders and the three label folders."""
    create_split_folders(features_folder_name, split_names)
    create_split_folders(labels_folder_name, split_names)

## 5.3. Splitting the Rows of One File

*How are the rows of one file divided?*

`train_test_split` cuts in two and 60/20/20 is three pieces, so it is used twice: first the 60%
training rows are taken off, then what is left is halved. The second `test_size` is
`test_size / (dev_size + test_size)` — half **of the remainder**, not half of the file, which is the
one piece of arithmetic in this section worth reading twice.

`stratify=` is what keeps each cut representative: sklearn draws per class, so a file that is 97%
Benign with 34 SQL Injection rows returns three pieces that are each 97% Benign, with 20, 7 and 7 SQL
Injection rows. The second call stratifies on the *remainder*, not on the original file, since that is
what it is cutting.

Rows are moved, columns are not touched, nothing is written: the three pieces are returned and 5.4
decides what goes where.

In [ ]:
def split_rows(
    df: pd.DataFrame,
    dev_size: float = DEV_SIZE,
    test_size: float = TEST_SIZE,
    label_column: str = LABEL_COLUMN,
    random_state: int = RANDOM_STATE,
) -> dict[str, pd.DataFrame]:
    """Split the rows of one file into train/dev/test, each keeping the class mix of the file."""
    held_out_size = dev_size + test_size
    if not 0.0 < held_out_size < 1.0:
        raise ValueError("dev_size + test_size must leave rows for training, so it must be < 1.0")

    train_rows, held_out_rows = train_test_split(
        df,
        test_size=held_out_size,
        random_state=random_state,
        stratify=df[label_column],
    )
    dev_rows, test_rows = train_test_split(
        held_out_rows,
        test_size=test_size / held_out_size,
        random_state=random_state,
        stratify=held_out_rows[label_column],
    )
    return {"train": train_rows, "dev": dev_rows, "test": test_rows}

## 5.4. Writing One Split

*Why are the features and the label written apart?*

Because almost every step after this one wants one or the other, not both. The imputer, the scaler and
the PCA only ever touch features; the label encoder and SMOTE only ever touch labels; a model
predicting on the test split must not be able to read its labels at all. Two trees make that
structural instead of a matter of remembering to drop a column.

The two trees are paired **by position**: the same file name holds the same rows in the same order in
both, and the row index is not written (`write_dataframe_to_parquet` writes `index=False`). That is
why both files are written here, side by side, from the same `rows` — and why nothing downstream may
reorder or drop rows in one tree without doing the same in the other.

In [ ]:
def write_split_to_parquet(
    rows: pd.DataFrame,
    split_name: str,
    file_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    label_column: str = LABEL_COLUMN,
) -> None:
    """Write one split of one file twice: its features and its label, under the same file name."""
    features = rows.drop(columns=[label_column])
    labels = rows[[label_column]]

    write_dataframe_to_parquet(
        features, get_split_folder(features_folder_name, split_name), file_name
    )
    write_dataframe_to_parquet(
        labels, get_split_folder(labels_folder_name, split_name), file_name
    )

## 5.5. Splitting One File

*How is one file handled from start to finish?*

Read once, split once, write six times — three splits $\times$ features and labels. This is the unit
of work a worker process runs in 5.6, and the only function in this section that both reads and
writes.

In [ ]:
def split_file(
    source_folder_name: str,
    file_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    dev_size: float = DEV_SIZE,
    test_size: float = TEST_SIZE,
    label_column: str = LABEL_COLUMN,
    random_state: int = RANDOM_STATE,
) -> None:
    """Split one Parquet file into train/dev/test and write each part as features and labels."""
    df = read_parquet_file(source_folder_name, file_name)
    rows_per_split = split_rows(df, dev_size, test_size, label_column, random_state)

    for split_name, rows in rows_per_split.items():
        write_split_to_parquet(
            rows, split_name, file_name, features_folder_name, labels_folder_name, label_column
        )

## 5.6. Splitting One Day

*How is a day run?*

The files of one day are the ones carrying that date in their name — 2.1.4 put it there — and each
becomes one task, the same shape as 2.1.6 and 4.3.

Working a day at a time is not something the arithmetic needs: every file is split on its own, so the
whole folder could be handed over in one go. It is kept because it turns the table above into a check.
If a date is missing from `data-no-inf/`, `split_one_day` raises instead of quietly producing a split
with a class missing from it, and a single day can be redone without touching the other nine.

In [ ]:
def split_one_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    day: str,
    dev_size: float = DEV_SIZE,
    test_size: float = TEST_SIZE,
    label_column: str = LABEL_COLUMN,
    random_state: int = RANDOM_STATE,
) -> None:
    """Split every Parquet file of one collection day, one worker process per file."""
    file_names = filter_file_names(
        get_all_file_names_in_folder(source_folder_name, "parquet"), day
    )
    if not file_names:
        raise ValueError(f"No Parquet files found for {day} in {source_folder_name}/.")

    create_feature_and_label_folders(features_folder_name, labels_folder_name)

    run_tasks_in_parallel(
        delayed(split_file)(
            source_folder_name,
            file_name,
            features_folder_name,
            labels_folder_name,
            dev_size,
            test_size,
            label_column,
            random_state,
        )
        for file_name in file_names
    )

    print(f"Split {len(file_names)} files of {day} into {', '.join(SPLIT_NAMES)}.")

## 5.7. Run the Split

Ten days, 168 files in, 1,008 files out: 168 $\times$ 3 splits $\times$ 2 trees. The days run one
after another, and the files within a day run in parallel.

In [ ]:
for day in COLLECTION_DAYS:
    split_one_day(
        PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_FEATURE, PATH_FOLDER_SPLIT_LABEL, day
    )

## 5.8. Check the Result

*Did every class survive in all three subsets?*

That is the question stratification was chosen to answer, so it is measured rather than assumed. Only
the label tree is read — one column per row instead of 78 — so the check costs a fraction of 5.7.

A class showing 0 in one of the three columns, or a share far from 60/20/20, would mean the split did
what a day-based split would have done, and every section after this one would be solving a different
problem than the one intended.

Counting the labels of a folder is done again in 7.9, on the encoded tree, so the two counting
functions live in section 1; what belongs here is only the comparison between the three splits.

In [ ]:
def summarize_splits(labels_folder_name: str = PATH_FOLDER_SPLIT_LABEL) -> pd.DataFrame:
    """Rows per class in each split, with the share of the class each split received."""
    counts_per_split = {
        split_name: count_labels_in_folder(
            get_split_folder(labels_folder_name, split_name), LABEL_COLUMN
        )
        for split_name in SPLIT_NAMES
    }

    table = pd.DataFrame(counts_per_split).fillna(0).astype(int)
    table["total"] = table[list(SPLIT_NAMES)].sum(axis=1)
    for split_name in SPLIT_NAMES:
        table[f"{split_name} %"] = (table[split_name] / table["total"] * 100).round(1)

    print(
        f"{table['total'].sum():,} rows over {len(table)} classes. "
        f"Target share: {TRAIN_SIZE:.0%} train / {DEV_SIZE:.0%} dev / {TEST_SIZE:.0%} test."
    )
    return table.sort_values("total", ascending=False)

In [ ]:
summarize_splits(PATH_FOLDER_SPLIT_LABEL)

16,232,943 rows went in and 16,232,943 came out — 9,739,760 train, 3,246,589 validation, 3,246,594
test — with all 15 classes present in all three subsets. Every class lands on 60.0 / 20.0 / 20.0 to
one decimal place, except the smallest: SQL Injection, 87 rows in the whole dataset, splits 52 / 18 /
17 (59.8 / 20.7 / 19.5). That is rounding inside individual files, not a defect of the split — with 87
rows there is no division into three that is exact.

The class names are the dataset's own and are kept exactly as they are, inconsistencies included:
`DDoS attacks-LOIC-HTTP` beside `DDOS attack-HOIC`, and `Infilteration` spelled the way the capture
tool spelled it. Nothing compares them by eye after this point — the label encoder maps each distinct
string to an integer — so tidying them would gain nothing and break the link back to the source files.
Note also that the two Infiltration days merge into a single class of 161,934 rows, and that
2018-02-20 contributes only `DDoS attacks-LOIC-HTTP`: the LOIC-UDP flows that the dataset description
places on that day appear on 2018-02-21 in the files themselves.

# 6. Imputation

A missing cell is not a number a model can multiply, so every one of them has to become a value
before anything is trained. This step fills them with the **median of their column, measured on the
training split only**, and writes the result as a new tree.

**In:** `data-split-feature/{train,dev,test}/*.parquet` $\rightarrow$
**Out:** `data-imputed/{train,dev,test}/*.parquet`, plus the fitted medians in
`cache/median-imputer.json`

`data-split-label/` is not touched: a row without a label was dropped back in 2.1.3, so there is
nothing missing there to fill.

### What is actually missing

191,520 cells, which is 0.015% of the feature matrix, and they are not scattered — all of them sit in
the two rate columns of section 4, and every one of them comes from the same 95,760 flows whose
duration is `0`:

| Column | Missing in `data-split-feature/` | Where it came from |
| --- | --- | --- |
| `flow pkts/s` | 95,760 | all 95,760 were $\infty$ in `data-raw/` (packets $\div$ 0) and became `NaN` in section 4 |
| `flow byts/s` | 95,760 | 36,039 were $\infty$ (bytes $\div$ 0); the other 59,721 were already `NaN` in `data-raw/` (0 bytes $\div$ 0) |

No other column has a single missing cell, and no column is missing everywhere — so a median exists
for all 78 of them.

### Why the median, and why only from the training split

The **median** rather than the mean, because flow features have extreme tails: one 5-minute flow next
to millions of millisecond flows drags a mean far away from any value the column actually takes,
while the median stays where the data is.

**Training split only**, because the validation and test rows are supposed to be data the pipeline has
never seen. A median taken over all the rows would carry a little of the test set into every imputed
cell of the training data, and the final numbers would be quietly optimistic. The same rule applies to
the scaler and the PCA later: fit on train, apply to all three.

### What this assumes

A zero-duration flow does not really have a "typical" packet rate — its true rate is undefined, and
this step replaces it with the most ordinary rate in the training data. That is a deliberate choice:
it keeps 95,760 otherwise complete rows (0.59% of the dataset, and the class mix of those rows is not
uniform) instead of dropping them, at the price of making them look average in exactly two columns. An
alternative would be a sentinel value or a "duration was zero" indicator column; both are worth trying
as a separate experiment.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 6.1 Settings | Where do the medians and the output go? | — |
| 6.2 Measuring the medians | How is a median taken over 9.7M rows? | 6.1 |
| 6.3 Storing them | Why are they written to a file at all? | 6.1 |
| 6.4 Fit | — | 6.1, 6.2, 6.3 |
| 6.5 Filling one file | What does the filling do? | — |
| 6.6 Filling every split | How are all three splits run? | 6.1, 6.3, 6.5 |
| 6.7 Run | — | 6.1, 6.6 |
| 6.8 Check the result | Is anything still missing? | 6.1 |

## 6.1. Settings

The input folder is `PATH_FOLDER_SPLIT_FEATURE` and the three split names are `SPLIT_NAMES`, both
from 5.1 — this step reads exactly what the split wrote.

`FIT_SPLIT` is the one that matters: it names the split every fitted number in this notebook is
allowed to come from. Writing it down once, here, is what keeps the leakage rule from becoming a
thing to remember.

`cache/` holds fitted artefacts rather than data — small files that are the *result* of looking at the
training split. JSON keeps the medians readable: 78 numbers anyone can open and check against the
table above.

`MEDIAN_FALLBACK` is only used for a column with no values at all in the training split, which cannot
have a median. It does not happen in this dataset (all 78 columns have one), and if it ever does, the
warning it prints is a signal to look at that column rather than to trust the 0.0.

In [ ]:
PATH_CACHE = "cache"
PATH_MEDIANS = os.path.join(PATH_CACHE, "median-imputer.json")

In [ ]:
PATH_FOLDER_IMPUTED = "data-imputed"

In [ ]:
FIT_SPLIT = "train"

In [ ]:
MEDIAN_FALLBACK = 0.0

## 6.2. Measuring the Medians

*How is a median taken over 9.7 million rows?*

A median is the one summary that cannot be accumulated file by file — the middle value of the whole
column is not the average of the middle values of its pieces — so this is the only step in the
notebook that has to look at a split as a whole.

It still does not have to *hold* it. `pl.scan_parquet` describes the files without reading them and
`collect` runs the reduction inside polars, so what crosses into Python is 78 numbers rather than 9.7
million rows. That is the entire reason polars appears here at all; the rest of the notebook is happy
with one pandas chunk at a time. Addressing the files of a split is a path question rather than an
imputation one, so `get_split_file_paths` sits in section 1 and 7.2 scans the label tree with it.

`replace_empty_medians` is separate because it answers a different question: not *what is the median*
but *what to do when there isn't one*. Keeping it out of `compute_medians` means the measurement stays
a measurement.

In [ ]:
def compute_medians(file_paths: list[str]) -> dict[str, float | None]:
    """Take one median per numeric column over all the given files, without loading them into Python.

    A column that is missing everywhere has no median, and polars returns `None` for it; 6.2's
    `replace_empty_medians` is what decides on that case.
    """
    lazy_frame = pl.scan_parquet(file_paths)
    schema = lazy_frame.collect_schema()

    numeric_columns = []
    for column, dtype in zip(schema.names(), schema.dtypes()):
        if dtype.is_numeric():
            numeric_columns.append(column)

    medians = lazy_frame.select(pl.col(numeric_columns).median()).collect(engine="streaming")
    return medians.row(0, named=True)

In [ ]:
def replace_empty_medians(
    medians: dict[str, float | None], fallback: float = MEDIAN_FALLBACK
) -> dict[str, float]:
    """Give a column that had nothing to measure a value to use, and say which columns those were."""
    complete_medians = {}
    empty_columns = []

    for column, median in medians.items():
        if median is None:
            empty_columns.append(column)
            complete_medians[column] = fallback
        else:
            complete_medians[column] = float(median)

    if empty_columns:
        print(
            f"Warning: {len(empty_columns)} columns had no value to take a median from and were "
            f"given {fallback}: {empty_columns}"
        )
    return complete_medians

## 6.3. Storing the Medians

*Why write them to a file instead of keeping the dictionary?*

Because the numbers outlive the notebook session. The workers in 6.6 read them, a later run of the
notebook can fill new data without re-reading the training split, and — most importantly for a thesis
— the file is the record of what was fitted. `cache/median-imputer.json` can be opened, quoted and
compared against a re-run.

It is also the boundary that keeps 6.2 honest: everything before it looks at training data, everything
after it only ever looks at this file.

In [ ]:
def save_medians(medians: dict[str, float], output_file: str) -> str:
    """Write the fitted medians as readable JSON; return the path they went to."""
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "w") as file:
        json.dump(medians, file, indent=4)
    return output_file

In [ ]:
def load_medians(input_file: str) -> dict[str, float]:
    """Read back the medians fitted earlier — the only thing 6.6 is allowed to know about training."""
    with open(input_file) as file:
        return json.load(file)

## 6.4. Fit the Imputer

Reads the training split of `data-split-feature/` once and leaves 78 numbers in
`cache/median-imputer.json`. This is the only cell in the section that touches training data.

In [ ]:
def create_median_imputer(
    source_folder_name: str, output_file: str, split_name: str = FIT_SPLIT
) -> dict[str, float]:
    """Measure the medians on one split, store them, and return what was stored."""
    file_paths = get_split_file_paths(source_folder_name, split_name)
    medians = compute_medians(file_paths)
    medians = replace_empty_medians(medians)
    save_medians(medians, output_file)

    print(
        f"Fitted {len(medians)} medians on the {split_name} split "
        f"({len(file_paths)} files) and saved them to {output_file}."
    )
    return medians

In [ ]:
medians = create_median_imputer(PATH_FOLDER_SPLIT_FEATURE, PATH_MEDIANS)
pd.Series(medians, name="median")

## 6.5. Filling One File

*What does the filling actually do?*

`fillna` with a dictionary fills each column with its own number in one pass, which is the whole
operation: no column is computed here, nothing is looked up from the data, the medians are simply
applied. A column the dictionary does not mention is left as it is, and a median for a column the file
does not have is ignored — so the same dictionary works for any file of the three splits.

Nothing about this depends on *which* split the file belongs to, which is exactly the point: the test
rows are filled by the same numbers as the training rows, and those numbers came only from training.

In [ ]:
def fill_missing_values(df: pd.DataFrame, medians: dict[str, float]) -> pd.DataFrame:
    """Replace every missing cell with the median of its column."""
    return df.fillna(medians)

In [ ]:
def impute_file(
    source_folder_name: str, target_folder_name: str, file_name: str, medians: dict[str, float]
) -> str:
    """Fill one Parquet file and write it to the target folder under the same name."""
    df = read_parquet_file(source_folder_name, file_name)
    df = fill_missing_values(df, medians)
    return write_dataframe_to_parquet(df, target_folder_name, file_name)

## 6.6. Filling Every Split

*How are all three splits run?*

Three folders instead of one, and inside each the familiar shape: one task per file, run in parallel.
Sections 7 and 8 rewrite a split tree in exactly the same way, so the walk itself lives in section 1
as `process_all_splits`; what is left here is what is specific to imputation.

The medians are loaded **once**, here, and handed to every worker. The splits are never distinguished
from this point on — train, dev and test are filled by the same 78 numbers, which is the property this
whole section exists to guarantee.

In [ ]:
def impute_all_splits(
    source_folder_name: str,
    target_folder_name: str,
    medians_file: str = PATH_MEDIANS,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> None:
    """Fill all three splits with the medians fitted on the training split."""
    medians = load_medians(medians_file)
    process_all_splits(
        source_folder_name, target_folder_name, split_names, impute_file, medians
    )

## 6.7. Run the Imputation

Reads `data-split-feature/` and fills `data-imputed/`, 504 files in and 504 out. The folder is a copy
with 191,520 cells changed, which is why it is written as a new tree rather than in place: if a median
turns out to be the wrong choice, 6.4 and 6.7 can be re-run without redoing the split.

In [ ]:
impute_all_splits(PATH_FOLDER_SPLIT_FEATURE, PATH_FOLDER_IMPUTED)

## 6.8. Check the Result

*Is anything still missing?*

The same count on the folder before and the folder after, as in 4.5 — and this one has a stricter
answer to give. An infinity that survived section 4 would only have been awkward; a missing cell that
survives this section will stop the scaler or the model outright, so "none left" has to be verified
rather than assumed.

The count is per column and per split, because a column that is still missing in `dev` but not in
`train` would mean the medians were fitted on a column that the other splits use differently.

In [ ]:
def count_missing_values_in_file(folder_name: str, file_name: str) -> pd.Series:
    """Count the missing cells per column in one Parquet file."""
    df = read_parquet_file(folder_name, file_name)
    return df.isna().sum()

In [ ]:
def count_missing_values_in_split(folder_name: str, split_name: str) -> pd.Series:
    """Count the missing cells per column across every file of one split."""
    split_folder = get_split_folder(folder_name, split_name)
    file_names = get_all_file_names_in_folder(split_folder, "parquet")

    counts_per_file = run_tasks_in_parallel(
        delayed(count_missing_values_in_file)(split_folder, file_name)
        for file_name in file_names
    )
    return sum(counts_per_file)

In [ ]:
def summarize_missing_values(
    folder_name: str, split_names: tuple[str, ...] = SPLIT_NAMES
) -> pd.DataFrame:
    """Missing cells per column in each split; columns with none are left out of the table."""
    counts_per_split = {
        split_name: count_missing_values_in_split(folder_name, split_name)
        for split_name in split_names
    }

    table = pd.DataFrame(counts_per_split)
    table["total"] = table[list(split_names)].sum(axis=1)

    print(f"{int(table['total'].sum()):,} missing cells in {folder_name}/.")
    return table[table["total"] > 0]

**Before the imputation:**

In [ ]:
summarize_missing_values(PATH_FOLDER_SPLIT_FEATURE)

**After the imputation:**

In [ ]:
summarize_missing_values(PATH_FOLDER_IMPUTED)

`data-split-feature/` is missing 191,520 cells: `flow byts/s` and `flow pkts/s`, 57,369 each in train,
19,235 each in dev and 19,156 each in test — the 60/20/20 shadow of the 95,760 zero-duration flows,
which is what a stratified split of those rows should look like.

`data-imputed/` returns an empty table: no column of any split has a missing cell left. The two
columns were filled with the training medians `flow byts/s` $= 786.0$ and `flow pkts/s` $= 136.2$
bytes and packets per second, and both are now safe to standardize in the next section — a mean and a
variance over a column with a `NaN` in it would have been `NaN` for the whole column.

# 7. Label Encoding

Every model in this notebook predicts a number, and `label` is a string. This step replaces the 15
class names with the integers 0–14, and keeps the names in a 15-line file so that any prediction can
be turned back into something a person can read.

**In:** `data-split-label/{train,dev,test}/*.parquet` $\rightarrow$
**Out:** `data-encoded-label/{train,dev,test}/*.parquet`, plus the class list in
`cache/label-encoder.json`

The column keeps its name, one file still corresponds to one file, and the rows keep their order, so
`data-encoded-label/` is a drop-in replacement for `data-split-label/` and the positional pairing with
the feature tree (5.4) survives untouched.

### The mapping

`LabelEncoder` sorts the class names and numbers them in order, so the ids fall out of the alphabet
rather than out of the data. Row counts are the ones 5.8 measured:

| id | class | rows |
| --- | --- | --- |
| 0 | Benign | 13,484,708 |
| 1 | Bot | 286,191 |
| 2 | Brute Force -Web | 611 |
| 3 | Brute Force -XSS | 230 |
| 4 | DDOS attack-HOIC | 686,012 |
| 5 | DDOS attack-LOIC-UDP | 1,730 |
| 6 | DDoS attacks-LOIC-HTTP | 576,191 |
| 7 | DoS attacks-GoldenEye | 41,508 |
| 8 | DoS attacks-Hulk | 461,912 |
| 9 | DoS attacks-SlowHTTPTest | 139,890 |
| 10 | DoS attacks-Slowloris | 10,990 |
| 11 | FTP-BruteForce | 193,360 |
| 12 | Infilteration | 161,934 |
| 13 | SQL Injection | 87 |
| 14 | SSH-Bruteforce | 187,589 |

Two things follow from that sort. `Benign` happens to come first, so class `0` is the benign class —
convenient for reading a confusion matrix, but an accident of the alphabet rather than a decision. And
the inconsistent capitalisation noted in 5.8 decides the order of the DDoS classes: `DDOS attack-HOIC`
takes 4 and 5 because an uppercase `O` sorts before the lowercase `o` of `DDoS attacks-LOIC-HTTP`.

The ids carry no meaning beyond identity — 14 is not further from 0 than 1 is. Every model here treats
them as categories, and the one place the numbering matters is that it must never change between
training and prediction, which is what storing it is for.

### Why it is still fitted on the training split

The same rule as the medians: a fitted number may only come from `FIT_SPLIT`. For a label encoder the
rule is free, because `fit` looks only at the *set* of names, not at how often they occur, and 5.8
showed that all 15 classes are in all three splits — measured again here, the three splits hold
exactly the same 15 distinct names, so fitting on train, on test or on everything would produce the
identical mapping.

Keeping the rule anyway buys a guard: if a future split ever drops a class from training,
`encoder.transform` raises on the split that still has it instead of quietly encoding it as something
else.

### Why a JSON class list and not a pickled encoder

A fitted `LabelEncoder` *is* its sorted list of classes — there is nothing else in it — so the list is
the artefact, and `LabelEncoder().fit(class_names)` rebuilds the encoder exactly. Storing the list as
JSON makes it readable (the table above is the file) and independent of the library version.

That is not hypothetical in this project: `cache/label-encoder.pkl`, left by an earlier run, was
pickled with scikit-learn 1.9.0 while this environment runs 1.6.1, so loading it warns that it "might
lead to breaking code or invalid results". Fifteen lines of JSON cannot go stale that way.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 7.1 Settings | Where do the ids and the class list go? | — |
| 7.2 Deciding the mapping | Which names become which ids? | 7.1 |
| 7.3 Storing the mapping | What exactly is saved, and how is it read back? | 7.2 |
| 7.4 Fit | — | 7.1, 7.2, 7.3 |
| 7.5 Encoding one file | What happens to a file? | — |
| 7.6 Encoding every split | How are all three run? | 7.1, 7.3, 7.5 |
| 7.7 Run | — | 7.1, 7.6 |
| 7.8 Decoding predictions | How does a prediction become a name again? | 7.1, 7.3 |
| 7.9 Check the result | Did the counts survive the encoding? | 7.1, 7.8 |

## 7.1. Settings

The input is `PATH_FOLDER_SPLIT_LABEL` (5.1), the split to fit on is `FIT_SPLIT` (6.1) and the column
is `LABEL_COLUMN` (2.1.1) — the constants the rest of the pipeline already uses. What is new is where
the ids and the class list go.

`cache/label-encoder.json` sits beside `cache/median-imputer.json` on purpose. Both are small, readable
records of something measured on the training split, and both are what the workers are handed instead
of the training data itself.

In [ ]:
PATH_LABEL_ENCODER = os.path.join(PATH_CACHE, "label-encoder.json")

In [ ]:
PATH_FOLDER_ENCODED_LABEL = "data-encoded-label"

## 7.2. Deciding the Mapping

*Which names become which ids?*

Only the distinct names are needed, so only the distinct names are read: `scan_parquet` and `unique`
hand back 15 strings instead of 9.7 million, the same trick 6.2 uses for the medians.

The order they come back in does not matter, because `create_label_encoder` is what decides the
order that counts — `LabelEncoder` sorts the names and numbers them. Splitting the two means the
reading can be checked without the numbering, and the numbering is one line with no I/O in it.

In [ ]:
def get_label_classes(
    labels_folder_name: str, split_name: str = FIT_SPLIT, label_column: str = LABEL_COLUMN
) -> list[str]:
    """Read the distinct class names of one split, without loading its rows into Python."""
    file_paths = get_split_file_paths(labels_folder_name, split_name)

    distinct_labels = (
        pl.scan_parquet(file_paths)
        .select(pl.col(label_column).unique())
        .collect(engine="streaming")
    )
    return distinct_labels[label_column].to_list()

In [ ]:
def create_label_encoder(class_names: list[str]) -> LabelEncoder:
    """Number the class names: `LabelEncoder` sorts them and hands out 0, 1, 2, ... in that order."""
    return LabelEncoder().fit(np.array(class_names))

## 7.3. Storing the Mapping

*What exactly is saved, and how is it read back?*

The list of class names, in the encoder's own order, so that the position of a name in the file **is**
its class id. Written from `encoder.classes_` rather than from the names that went in, the file cannot
disagree with the encoder that produced it.

Reading it back re-fits an encoder on that list. Because `fit` only sorts and numbers, and the list is
already in that order, the rebuilt encoder is the same mapping — no pickle, no version to match.

Reading the *file* and rebuilding the *encoder* are two different needs: 10.6 wants the names to label
a table and has no use for sklearn, so `load_label_classes` lives in section 1 and the encoder is what
this section adds on top of it.

In [ ]:
def save_label_classes(class_names: list[str], output_file: str) -> str:
    """Write the class list as JSON; a name's position in the list is its class id."""
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "w") as file:
        json.dump(class_names, file, indent=4)
    return output_file

In [ ]:
def load_label_encoder(input_file: str) -> LabelEncoder:
    """Rebuild the encoder from the stored class list — this is all 7.6 and 7.8 are given."""
    return create_label_encoder(load_label_classes(input_file))

## 7.4. Fit the Encoder

Reads the distinct labels of the training split and leaves 15 names in `cache/label-encoder.json`.
This is the only cell in the section that looks at training data, and the table it prints is the one
reproduced at the top of the section.

In [ ]:
def create_and_save_label_encoder(
    labels_folder_name: str, output_file: str, split_name: str = FIT_SPLIT
) -> LabelEncoder:
    """Take the class names from one split, store them in encoder order, and return the encoder."""
    class_names = get_label_classes(labels_folder_name, split_name)
    encoder = create_label_encoder(class_names)
    save_label_classes([str(class_name) for class_name in encoder.classes_], output_file)

    print(
        f"Found {len(encoder.classes_)} classes in the {split_name} split "
        f"and saved them to {output_file}."
    )
    return encoder

In [ ]:
label_encoder = create_and_save_label_encoder(PATH_FOLDER_SPLIT_LABEL, PATH_LABEL_ENCODER)
pd.Series(label_encoder.classes_, name="class").rename_axis("id").to_frame()

## 7.5. Encoding One File

*What happens to a file?*

A label file holds one column and nothing else, so encoding it is one call. The frame is rebuilt
rather than edited in place because the column changes type — strings out, integers in — and the name
is kept so that nothing downstream needs to know whether it is reading names or ids.

`transform` raises on a name it has never seen. That is worth keeping rather than guarding against: it
is the only thing standing between a stray label and a silently wrong class id.

In [ ]:
def encode_labels(
    labels: pd.DataFrame, encoder: LabelEncoder, label_column: str = LABEL_COLUMN
) -> pd.DataFrame:
    """Replace the class names with their ids, keeping the column name and the row order."""
    return pd.DataFrame({label_column: encoder.transform(labels[label_column])})

In [ ]:
def encode_label_file(
    source_folder_name: str, target_folder_name: str, file_name: str, encoder: LabelEncoder
) -> str:
    """Encode one label file and write it to the target folder under the same name."""
    labels = read_parquet_file(source_folder_name, file_name)
    encoded_labels = encode_labels(labels, encoder)
    return write_dataframe_to_parquet(encoded_labels, target_folder_name, file_name)

## 7.6. Encoding Every Split

*How are all three run?*

The same `process_all_splits` as 6.6, with an encoder in place of a dictionary of medians: the mapping
is read once, the three folders are created, and then one task per file.

All three splits are encoded with the same mapping, which is the property this section exists to
guarantee. A dev file encoded by a mapping of its own would put different attacks behind the same
number, and every report built from it would be wrong in a way that is almost impossible to see.

In [ ]:
def encode_all_splits(
    source_folder_name: str,
    target_folder_name: str,
    encoder_file: str = PATH_LABEL_ENCODER,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> None:
    """Encode the labels of all three splits with the mapping fitted on the training split."""
    encoder = load_label_encoder(encoder_file)
    process_all_splits(
        source_folder_name, target_folder_name, split_names, encode_label_file, encoder
    )

## 7.7. Run the Encoding

Reads `data-split-label/` and fills `data-encoded-label/`, 504 files in and 504 out. The cheapest step
in the notebook: one column of short strings per row, and the whole mapping is 15 entries.

In [ ]:
encode_all_splits(PATH_FOLDER_SPLIT_LABEL, PATH_FOLDER_ENCODED_LABEL)

## 7.8. Turning Predictions Back Into Names

*How does a prediction become a name again?*

A model predicts `8`; a report has to say `DoS attacks-Hulk`. `decode_labels` is that direction, and
it takes the path of the class list rather than an encoder, so a section far below can decode without
carrying an encoder along with it.

This is what the stored mapping is ultimately for. Without it the encoded tree is 15 anonymous
integers, and every classification report and confusion matrix built later would be unreadable.

In [ ]:
def decode_labels(encoded, encoder_file: str = PATH_LABEL_ENCODER) -> np.ndarray:
    """Turn class ids back into class names — for a report, a confusion matrix, or 7.9 below."""
    encoder = load_label_encoder(encoder_file)
    return encoder.inverse_transform(np.asarray(encoded))

## 7.9. Check the Result

*Did the counts survive the encoding?*

Encoding has to be a renaming and nothing more, so the test is that the table below matches the one
5.8 printed, row for row, with ids in place of names. Each id is decoded back through 7.8 to put the
name beside it, which means a mapping saved wrong, read back wrong or applied wrong shows up here as a
mismatched count instead of as a quietly wrong model several sections later.

In [ ]:
def summarize_encoded_labels(
    folder_name: str,
    encoder_file: str = PATH_LABEL_ENCODER,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> pd.DataFrame:
    """Rows per class id in each split, with the name each id stands for."""
    counts_per_split = {
        split_name: count_labels_in_folder(
            get_split_folder(folder_name, split_name), LABEL_COLUMN
        )
        for split_name in split_names
    }

    table = pd.DataFrame(counts_per_split).fillna(0).astype(int).sort_index()
    table.insert(0, "class", decode_labels(table.index, encoder_file))
    table["total"] = table[list(split_names)].sum(axis=1)

    print(f"{table['total'].sum():,} rows over {len(table)} classes in {folder_name}/.")
    return table

In [ ]:
summarize_encoded_labels(PATH_FOLDER_ENCODED_LABEL)

16,232,943 rows over 15 ids, `0` to `14`, and the per-class counts are the ones 5.8 printed for the
names: 13,484,708 rows of id 0 (`Benign`), 87 of id 13 (`SQL Injection`), and every class in all three
splits. Nothing was merged, dropped or renumbered, and every id decoded back to the name it came from.

The feature tree is now numbers, the label tree is now numbers, and both are still paired by file name
and row order. What is left before training is scaling the features — which is where the training-split
rule of 6.1 is used for the third time.

# 8. Feature Scaling

The 78 features are measured on scales that have nothing to do with one another. Taken from the
imputed training split, the column standard deviations run from `0.0129` (`cwe flag count`) to
`783,709,249` (`fwd iat min`) — a factor of sixty billion — and the means from `0` to `11,671,527`.

Left alone, that decides the outcome of anything that adds features together or measures a distance
between two rows. KNN would be a nearest-neighbour search on `flow duration` with 77 rounding errors
attached, gradient descent would crawl along the widest axis, and the PCA of section 9 — which looks
for the directions of greatest variance — would return `flow duration` as its first component every
time, because variance measured in microseconds dwarfs variance measured in flags.

Standardizing puts every column on the same footing. Subtract the column's mean, divide by its
standard deviation, and each feature arrives as *how many standard deviations from typical this row
is*:

$$z = \frac{x - \mu}{\sigma}$$

**In:** `data-imputed/{train,dev,test}/*.parquet` $\rightarrow$
**Out:** `data-scaled/{train,dev,test}/*.parquet`, plus $\mu$ and $\sigma$ per column in
`cache/standard-scaler.json`

The Random Forest and the XGBoost of the later sections do not need this — a tree splits one column at
a time and never compares units — but they are not harmed by it either, and running every model on the
same matrix keeps the comparison between them about the models.

### $\mu$ and $\sigma$ from the training split only

The rule of 6.1, for the third time: the mean and the standard deviation are measurements of the
training data, and letting dev or test rows into them would decide how every training row is
represented using data the model is supposed to be judged on.

The consequence is visible, and worth expecting rather than being surprised by. After scaling, the
training split has mean 0 and standard deviation 1 by construction, while dev and test land *near*
them without hitting them — 8.8 measures exactly that. A dev split that came out at exactly 0 and 1
would be the evidence that it had been scaled with its own statistics, which is the mistake this
section is arranged to prevent.

### Eight columns that carry nothing

Eight of the 78 features never change in the training split — every row holds `0`:

`bwd psh flags`, `bwd urg flags`, `fwd byts/b avg`, `fwd pkts/b avg`, `fwd blk rate avg`,
`bwd byts/b avg`, `bwd pkts/b avg`, `bwd blk rate avg`

A constant column has a standard deviation of `0`, and dividing by it would produce the infinities
section 4 spent its time removing. `StandardScaler` stores `1.0` in place of that `0`, so the column
becomes `0` everywhere and stays finite. That is why 8.3 saves `scale_` rather than the standard
deviation it came from: the substitution is part of the fitted state, not an afterthought.

They are left in the data rather than dropped. An all-zero column adds no variance, so the PCA of
section 9 has nothing to extract from it, and removing it here would mean carrying a second list of
columns through every step that follows.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 8.1 Settings | Where do the scaled files and the statistics go? | — |
| 8.2 Measuring $\mu$ and $\sigma$ | How, over 9.7 million rows? | 8.1 |
| 8.3 Storing them | What is saved, and why `scale_` and not $\sigma$? | — |
| 8.4 Fit | — | 8.1, 8.2, 8.3 |
| 8.5 Scaling one file | What is applied to a file? | 2.1.1 |
| 8.6 Scaling every split | How are all three run? | 8.1, 8.3, 8.5 |
| 8.7 Run | — | 8.1, 8.6 |
| 8.8 Check the result | Did train land on 0 and 1, and dev/test near them? | 8.1 |

## 8.1. Settings

The input is `PATH_FOLDER_IMPUTED` (6.1) — scaling comes after imputation because a mean cannot be
taken over missing cells — and the split to fit on is `FIT_SPLIT` (6.1) again.

`cache/standard-scaler.json` is the third artefact of the same kind, next to the medians and the class
list: something measured on the training split, small enough to read, and the only thing the workers
are given.

In [ ]:
PATH_SCALER = os.path.join(PATH_CACHE, "standard-scaler.json")

In [ ]:
PATH_FOLDER_SCALED = "data-scaled"

## 8.2. Measuring the Mean and the Standard Deviation

*How, over 9.7 million rows?*

`partial_fit` is the answer, and it is why this section does not need the lazy scan that 6.2 needed. A
mean and a variance can be *accumulated*: `StandardScaler` keeps a running count, sum and sum of
squares, and every file updates them. Memory stays at one file, the file order does not change the
result, and the result matches a fit on everything at once up to floating-point rounding.

This is the one loop in the notebook that is not run in parallel, and it cannot be — every file
updates the same running state, so the files have to be taken one after another. It is also why the
loop is plain: there is nothing here to distribute.

The empty-file guard is there because `partial_fit` raises on a frame with no rows rather than
ignoring it. No file in this dataset is empty; the guard costs a line and turns a crash into a skipped
file. `rows_seen` then replaces the original's "was anything fitted at all?" check with the number
that actually answers it.

In [ ]:
def fit_scaler(source_folder_name: str, split_name: str = FIT_SPLIT) -> StandardScaler:
    """Accumulate the mean and variance of every column, one file at a time."""
    split_folder = get_split_folder(source_folder_name, split_name)
    file_names = get_all_file_names_in_folder(split_folder, "parquet")

    scaler = StandardScaler()
    rows_seen = 0
    for file_name in file_names:
        df = read_parquet_file(split_folder, file_name)
        if df.empty:
            continue
        scaler.partial_fit(df)
        rows_seen += len(df)

    if rows_seen == 0:
        raise ValueError(f"No rows to fit a scaler on in {split_folder}/.")

    print(f"Fitted on {rows_seen:,} rows from {len(file_names)} files in {split_folder}/.")
    return scaler

## 8.3. Storing the Statistics

*What is saved, and why `scale_` and not $\sigma$?*

Two numbers per column — the mean to subtract and the number to divide by — written as JSON beside the
medians and the class list, for the same reasons: readable, quotable, and not tied to a library
version.

`scale_` is the divisor `StandardScaler` actually uses, which is $\sigma$ everywhere except the eight
constant columns, where it is `1.0` instead of `0`. Saving $\sigma$ and dividing by it later would
reintroduce the division by zero that sklearn had already dealt with, so what is saved is the divisor,
not the measurement it came from.

The column names come from the scaler itself (`feature_names_in_`, set when it was fitted on a
`DataFrame`), so the file cannot disagree with the object that produced it.

In [ ]:
def save_scaler_statistics(scaler: StandardScaler, output_file: str) -> str:
    """Write the fitted mean and divisor of every column as JSON; return the path they went to."""
    column_names = [str(name) for name in scaler.feature_names_in_]
    statistics = {
        "mean": {name: float(mean) for name, mean in zip(column_names, scaler.mean_)},
        "scale": {name: float(scale) for name, scale in zip(column_names, scaler.scale_)},
    }

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "w") as file:
        json.dump(statistics, file, indent=4)
    return output_file

In [ ]:
def load_scaler_statistics(input_file: str) -> dict[str, dict[str, float]]:
    """Read back the mean and divisor of every column — all 8.6 is given about the training split."""
    with open(input_file) as file:
        return json.load(file)

## 8.4. Fit the Scaler

Reads the training split of `data-imputed/` once and leaves 78 means and 78 divisors in
`cache/standard-scaler.json`. The table it prints is the scaling itself: the size of the numbers in
the `mean` and `scale` columns is the spread the introduction describes.

In [ ]:
def create_and_save_scaler(
    source_folder_name: str, output_file: str, split_name: str = FIT_SPLIT
) -> StandardScaler:
    """Fit the scaler on one split, store its statistics, and return the fitted scaler."""
    scaler = fit_scaler(source_folder_name, split_name)
    save_scaler_statistics(scaler, output_file)

    print(f"Saved {len(scaler.mean_)} means and divisors to {output_file}.")
    return scaler

In [ ]:
scaler = create_and_save_scaler(PATH_FOLDER_IMPUTED, PATH_SCALER)
pd.DataFrame(load_scaler_statistics(PATH_SCALER))

## 8.5. Scaling One File

*What is applied to a file?*

The subtraction and the division, written out. `scaler.transform` would do the same arithmetic, but
doing it with two pandas Series means the numbers come from the JSON file rather than from a live
object, and what happens to the data stays legible: `(df - means) / scales`, aligned by column name.

The column check is there because pandas alignment would otherwise hide a mistake. Subtracting a
Series that is missing a column does not raise — it quietly produces a column of `NaN`, which would
travel all the way into a model. Comparing the two sets of names first turns that into an error at the
first file instead of a puzzle at the last one.

The result is cast back to `FEATURE_DTYPE` (2.1.1): dividing `float32` by a `float64` statistic gives
`float64`, which would double the size of `data-scaled/` for digits that carry no information.

In [ ]:
def scale_features(df: pd.DataFrame, statistics: dict[str, dict[str, float]]) -> pd.DataFrame:
    """Standardize every column: subtract its mean, divide by its divisor."""
    means = pd.Series(statistics["mean"])
    scales = pd.Series(statistics["scale"])

    if set(df.columns) != set(means.index):
        raise ValueError("this file's columns are not the columns the scaler was fitted on")

    return ((df - means) / scales).astype(FEATURE_DTYPE)

In [ ]:
def scale_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    statistics: dict[str, dict[str, float]],
) -> str:
    """Scale one Parquet file and write it to the target folder under the same name."""
    df = read_parquet_file(source_folder_name, file_name)
    scaled_features = scale_features(df, statistics)
    return write_dataframe_to_parquet(scaled_features, target_folder_name, file_name)

## 8.6. Scaling Every Split

*How are all three run?*

`process_all_splits` again, the same one 6.6 and 7.6 use. What is specific to scaling is one line: the
statistics are read once, and every worker gets the same ones.

That is the sentence worth keeping from all three sections. The imputer, the encoder and the scaler are
each fitted on the training split and then applied, unchanged, to all three — so whatever the models
are eventually measured on, the numbers describing it were never allowed to shape it.

In [ ]:
def scale_all_splits(
    source_folder_name: str,
    target_folder_name: str,
    statistics_file: str = PATH_SCALER,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> None:
    """Scale all three splits with the statistics fitted on the training split."""
    statistics = load_scaler_statistics(statistics_file)
    process_all_splits(
        source_folder_name, target_folder_name, split_names, scale_file, statistics
    )

## 8.7. Run the Scaling

Reads `data-imputed/` and fills `data-scaled/`, 504 files in and 504 out. The output is smaller than
the input — `float32` in place of the `float64` the arithmetic produces — and it is the last tree the
features pass through before the PCA of section 9.

In [ ]:
scale_all_splits(PATH_FOLDER_IMPUTED, PATH_FOLDER_SCALED)

## 8.8. Check the Result

*Did train land on 0 and 1, and dev/test near them?*

Run on the folder before and the folder after, like 4.5 and 6.8, except that here the two tables are
not "many" and "none" but "every scale imaginable" and "one scale".

Per column would be 78 rows three times over, and the interesting part is the extremes, so the table
reports the worst offender in each split: the column furthest from mean 0, and the range the standard
deviations fall in. For the training split those have to read 0 and 1. For dev and test they have to
read *close to* 0 and 1 — close enough that the same transformation clearly applies to them, far
enough to show that their own statistics were never consulted.

Measuring the columns of a split is done again in 9.9, on the projected tree, so
`measure_feature_statistics` lives in section 1; the summary below is what is specific to scaling.

In [ ]:
def summarize_scaling(
    folder_name: str, split_names: tuple[str, ...] = SPLIT_NAMES
) -> pd.DataFrame:
    """How far from mean 0 and standard deviation 1 each split is, and which column is furthest."""
    summary = {}
    for split_name in split_names:
        statistics = measure_feature_statistics(folder_name, split_name)
        summary[split_name] = {
            "columns": len(statistics),
            "largest |mean|": statistics["mean"].abs().max(),
            "at column": statistics["mean"].abs().idxmax(),
            "smallest std": statistics["std"].min(),
            "largest std": statistics["std"].max(),
            "at column ": statistics["std"].idxmax(),
        }

    print(f"Measured {folder_name}/.")
    return pd.DataFrame(summary).T

**Before the scaling:**

In [ ]:
summarize_scaling(PATH_FOLDER_IMPUTED)

**After the scaling:**

In [ ]:
summarize_scaling(PATH_FOLDER_SCALED)

Before scaling, the training split's column means reach `11,671,527` (`flow duration`) and its
standard deviations `783,709,249` (`fwd iat min`). After it, the largest mean anywhere in the training
split is $1.8 \times 10^{-8}$, and every standard deviation reads `1` to the precision the table
shows — except the eight constant
columns, which are `0`, as intended.

Dev and test land near those numbers without reaching them: largest mean `0.0027` in dev and `0.0013`
in test, standard deviations up to `5.70` and `1.15`. That gap is not an error to be corrected — it is
the evidence that both were scaled with the *training* split's statistics. A column that spreads 5.7
times wider in dev than in train is an honest measurement of how the two samples differ, and it is
exactly the kind of shift a deployed model meets.

# 9. Incremental Principal Component Analysis

Standardizing made the 78 features comparable; it did not make them independent. CICFlowMeter measures
the same flow many times over — `fwd pkt len max` beside `fwd pkt len mean` beside `pkt len max`,
`flow iat min` beside `fwd iat min` — so the columns move together and much of what they say is said
more than once.

PCA rotates those 78 axes into new ones that are uncorrelated by construction and ordered by how much
of the spread they account for, and keeps the first few. After 8.7 each of the 70 varying columns
carries exactly one unit of variance and the eight constant ones carry zero, so there are **70 units**
to distribute — and the first **24** new axes hold **95.5%** of them.

**In:** `data-scaled/{train,dev,test}/*.parquet` $\rightarrow$
**Out:** `data-ipca/{train,dev,test}/*.parquet` with columns `pc1 … pc24`, plus the projection in
`cache/ipca.npz`

The gain is not disk space but the models. KNN computes a distance across every column of every row
and an SVM kernel does the same, so 78 columns down to 24 takes a third of the arithmetic out of the
two most expensive models in the study — and the discarded axes are mostly the redundancy above, not
information.

### Why *incremental*

`PCA` wants the whole matrix in memory for its SVD: 9.7 million rows of 78 `float32` columns is about
3 GB before the copies a decomposition makes. `IncrementalPCA` takes the same `partial_fit` route as
the scaler in 8.2 — each file updates a running basis — so memory stays at one file. The fit below
takes under half a minute.

### One fit, not eight

Fitting a separate transformer for each candidate size — 5, 10, … 40 — is eight passes over the
training split for something one pass already contains. The components come out **ordered by
variance**, so the first $k$ rows of a 40-component fit *are* a $k$-component projection, and the
cumulative sum of `explained_variance_ratio_` is the whole curve at once.

Measured on this data, the single fit is not only cheaper but strictly better. `IncrementalPCA` carries
only `n_components` directions from one file to the next, so a narrow fit throws more away at every
step:

| components | one 40-component fit | eight separate fits |
| --- | --- | --- |
| 5 | 53.1% | 49.5% |
| 10 | 73.6% | 71.2% |
| 15 | 84.8% | 82.5% |
| 20 | 91.6% | 88.9% |
| 25 | 96.3% | 95.3% |
| 30 | 98.8% | 98.7% |
| 35 | 99.71% | 99.69% |
| 40 | 99.92% | 99.92% |

The two agree at 40, where they are the same fit, and diverge below it in the direction the argument
predicts. (The right-hand column comes from the eight transformers in `cache/pca/`, left by the
earlier pipeline.)

### How many components

The rule is written down instead of the number: **the fewest components whose cumulative variance
reaches `VARIANCE_TARGET`**. At 95% that is **24** components, keeping 95.5%; 23 falls just short at
94.7%.

The earlier pipeline used 25 under the same rule — 25 was the smallest of the eight *evaluated* sizes
to clear 95%, and a grid of multiples of five cannot see 24. Both satisfy the target. This notebook
takes the smallest the curve allows and prints it in 9.5 rather than asserting it in a settings cell;
if you would rather keep 25 for continuity with models already trained on 25 components, replace the
derived value with `IPCA_COMPONENTS = 25` and nothing else changes.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 9.1 Settings | Where does the projection go, and how much variance is enough? | — |
| 9.2 Fitting the projection | How, over 9.7 million rows? | 9.1 |
| 9.3 Storing it | What is saved, and in what format? | — |
| 9.4 Fit | — | 9.1, 9.2, 9.3 |
| 9.5 Choosing the number of components | How many are enough? | 9.1, 9.3 |
| 9.6 Projecting one file | What is applied to a file? | 2.1.1 |
| 9.7 Projecting every split | How are all three run? | 9.1, 9.3, 9.6 |
| 9.8 Run | — | 9.5, 9.7 |
| 9.9 Check the result | Did the projection keep what the curve promised? | 9.1, 9.3 |

## 9.1. Settings

The input is `PATH_FOLDER_SCALED` (8.1) — the projection has to come after standardizing, because PCA
follows variance and unstandardized variance is just a choice of units — and the split to fit on is
`FIT_SPLIT` (6.1) for the fourth and last time.

`MAX_COMPONENTS` is how far the curve is drawn, not how many are used. It costs one fit to have the
whole curve, so it is set well above any plausible answer; the only thing it must not do is fall below
the number the target ends up asking for, which 9.5 would then refuse.

`VARIANCE_TARGET` is the actual decision of this section: how much of the training variance the
projection has to keep. 95% is the conventional line and the one the earlier pipeline drew.

In [ ]:
PATH_IPCA = os.path.join(PATH_CACHE, "ipca.npz")

In [ ]:
PATH_FOLDER_IPCA = "data-ipca"

In [ ]:
MAX_COMPONENTS = 40

In [ ]:
VARIANCE_TARGET = 0.95

## 9.2. Fitting the Projection

*How, over 9.7 million rows?*

The same shape as 8.2: a running state that every file updates, and the one other loop in the notebook
that cannot be parallel. `partial_fit` re-decomposes the basis it is carrying together with the new
file, so the files have to arrive one after another.

The guard is stricter here than the scaler's. `IncrementalPCA` needs **at least `n_components` rows**
in a batch to extract that many directions from it, so a file shorter than the basis is skipped rather
than being allowed to raise. No file in this dataset is anywhere near that small — the shortest is tens
of thousands of rows — but the guard is what makes the function safe to point at a folder of small
files.

In [ ]:
def fit_ipca(
    source_folder_name: str,
    split_name: str = FIT_SPLIT,
    n_components: int = MAX_COMPONENTS,
) -> IncrementalPCA:
    """Accumulate the principal components of a split, one file at a time."""
    split_folder = get_split_folder(source_folder_name, split_name)
    file_names = get_all_file_names_in_folder(split_folder, "parquet")

    ipca = IncrementalPCA(n_components=n_components)
    rows_seen = 0
    for file_name in file_names:
        df = read_parquet_file(split_folder, file_name)
        if len(df) < n_components:
            continue
        ipca.partial_fit(df)
        rows_seen += len(df)

    if rows_seen == 0:
        raise ValueError(
            f"No file in {split_folder}/ has the {n_components} rows a fit of this size needs."
        )

    print(f"Fitted {n_components} components on {rows_seen:,} rows from {split_folder}/.")
    return ipca

## 9.3. Storing the Projection

*What is saved, and in what format?*

Four arrays, and they are the whole transformation:

| array | shape | what it is |
| --- | --- | --- |
| `components` | 40 × 78 | the new axes, each a direction through the old 78, ordered by variance |
| `mean` | 78 | the centre the rows are measured from |
| `explained_variance` | 40 | how much variance each axis carries |
| `explained_variance_ratio` | 40 | the same as a share of the 70 available units |
| `columns` | 78 | the column order the matrix was built for |

The medians and the scaler statistics were saved as JSON because 78 numbers can be read; a 40 × 78
matrix cannot, so this one is `.npz` — still plain arrays with no pickled class in them, so it does not
depend on the scikit-learn version the way `cache/pca/*.pkl` does.

`columns` is stored for the same reason 8.5 compares column names, but the check it enables is
stricter: a matrix multiplication is **positional**, so here the *order* is part of the contract, not
just the set of names.

In [ ]:
def save_ipca(ipca: IncrementalPCA, output_file: str) -> str:
    """Store the fitted projection as plain arrays; return the path they went to."""
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    np.savez(
        output_file,
        components=ipca.components_,
        mean=ipca.mean_,
        explained_variance=ipca.explained_variance_,
        explained_variance_ratio=ipca.explained_variance_ratio_,
        columns=np.array([str(name) for name in ipca.feature_names_in_]),
    )
    return output_file

In [ ]:
def load_ipca(input_file: str) -> dict[str, np.ndarray]:
    """Read the stored projection back as arrays — all 9.5 and 9.7 are given."""
    with np.load(input_file) as stored:
        return {name: stored[name] for name in stored.files}

## 9.4. Fit the Projection

Reads the training split of `data-scaled/` once and leaves five arrays in `cache/ipca.npz`. This is
the last cell in the notebook that looks at training data in order to decide something.

In [ ]:
def create_and_save_ipca(
    source_folder_name: str,
    output_file: str,
    split_name: str = FIT_SPLIT,
    n_components: int = MAX_COMPONENTS,
) -> IncrementalPCA:
    """Fit the projection on one split, store its arrays, and return the fitted transformer."""
    ipca = fit_ipca(source_folder_name, split_name, n_components)
    save_ipca(ipca, output_file)

    print(
        f"Saved {n_components} components of {ipca.n_features_in_} features to {output_file}."
    )
    return ipca

In [ ]:
ipca = create_and_save_ipca(PATH_FOLDER_SCALED, PATH_IPCA)

## 9.5. Choosing the Number of Components

*How many are enough?*

The curve is a cumulative sum and the choice is the first index that reaches the target — two lines of
arithmetic on an array that 9.4 already wrote. Nothing is re-fitted, which is the payoff of fitting
once at `MAX_COMPONENTS`.

The plot is here because the number alone hides the shape: the first two components carry 31% between
them, the curve is still climbing steeply at 10, and past 30 it is nearly flat, so the target is
choosing a point on a bend rather than a cliff. A different target would move the answer by a few
components, and the curve is what makes that visible.

`IPCA_COMPONENTS` is defined here rather than in 9.1 on purpose. It is not a decision — it is what
`VARIANCE_TARGET` and the curve amount to together, and a settings cell asserting `25` is exactly the
number a reader cannot check.

In [ ]:
def get_variance_curve(input_file: str = PATH_IPCA) -> pd.Series:
    """Cumulative share of the training variance kept by the first 1, 2, 3, ... components."""
    explained_variance_ratio = load_ipca(input_file)["explained_variance_ratio"]

    return pd.Series(
        np.cumsum(explained_variance_ratio),
        index=range(1, len(explained_variance_ratio) + 1),
        name="cumulative variance",
    )

In [ ]:
def choose_component_count(
    variance_curve: pd.Series, variance_target: float = VARIANCE_TARGET
) -> int:
    """The fewest components whose cumulative variance reaches the target."""
    reaching_target = variance_curve[variance_curve >= variance_target]
    if reaching_target.empty:
        raise ValueError(
            f"{len(variance_curve)} components reach only {variance_curve.iloc[-1]:.1%}; "
            f"raise MAX_COMPONENTS or lower the {variance_target:.0%} target."
        )
    return int(reaching_target.index[0])

In [ ]:
def plot_variance_curve(
    variance_curve: pd.Series, variance_target: float = VARIANCE_TARGET
) -> None:
    """Draw the cumulative variance against the number of components, marking the chosen count."""
    surface, grid, ink, muted, series = "#fcfcfb", "#e8e7e3", "#0b0b0b", "#52514e", "#2a78d6"
    chosen = choose_component_count(variance_curve, variance_target)

    figure, axes = plt.subplots(figsize=(8, 4.5))
    figure.patch.set_facecolor(surface)
    axes.set_facecolor(surface)

    axes.plot(variance_curve.index, variance_curve * 100, color=series, linewidth=2)
    axes.axhline(variance_target * 100, color=muted, linewidth=1, linestyle=(0, (4, 3)))
    axes.text(1, variance_target * 100 + 2, f"{variance_target:.0%} target", color=muted, fontsize=9)

    axes.plot(
        [chosen], [variance_curve[chosen] * 100], marker="o", markersize=8,
        color=series, markeredgecolor=surface, markeredgewidth=2, zorder=3,
    )
    axes.annotate(
        f"{chosen} components → {variance_curve[chosen]:.1%}",
        (chosen, variance_curve[chosen] * 100),
        xytext=(10, -18), textcoords="offset points", color=ink, fontsize=9,
    )

    axes.set_title(
        "Variance retained by the first n principal components", color=ink, fontsize=11, loc="left", pad=12
    )
    axes.set_xlabel("Principal components", color=muted, fontsize=9)
    axes.set_ylabel("Cumulative explained variance (%)", color=muted, fontsize=9)
    axes.set_xlim(0, variance_curve.index[-1])
    axes.set_ylim(0, 100)
    axes.set_xticks(range(0, variance_curve.index[-1] + 1, 5))
    axes.grid(axis="y", color=grid, linewidth=1)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axes.spines[side].set_color(grid)
    axes.tick_params(colors=muted, labelsize=9, length=0)

    figure.tight_layout()
    plt.show()

In [ ]:
variance_curve = get_variance_curve(PATH_IPCA)
plot_variance_curve(variance_curve)

In [ ]:
IPCA_COMPONENTS = choose_component_count(variance_curve)
print(
    f"{IPCA_COMPONENTS} of {len(variance_curve)} components keep "
    f"{variance_curve[IPCA_COMPONENTS]:.2%} of the training variance."
)

## 9.6. Projecting One File

*What is applied to a file?*

Subtract the centre, multiply by the first `n_components` axes. Written out for the same reason 8.5 is:
the numbers come from the stored arrays rather than from a live object, and `(rows - mean) @ axes.T` is
the whole transformation in one line.

Slicing `components[:n_components]` is where the "one fit, not eight" argument is actually cashed in —
the rows of the matrix are ordered by variance, so taking the first 24 of 40 is the 24-component
projection.

The columns come out named `pc1 … pc24` and mean something different from everything before them: not
a packet count or a duration, but a direction through all 78 at once. Nothing downstream reads them by
name, and the row order is untouched, so the pairing with `data-encoded-label/` still holds.

In [ ]:
def project_features(
    df: pd.DataFrame, projection: dict[str, np.ndarray], n_components: int
) -> pd.DataFrame:
    """Rotate the rows onto the first `n_components` principal components."""
    if list(df.columns) != list(projection["columns"]):
        raise ValueError(
            "this file's columns are not, in order, the columns the projection was fitted on"
        )

    axes = projection["components"][:n_components]
    projected = (df.to_numpy() - projection["mean"]) @ axes.T

    column_names = [f"pc{number}" for number in range(1, n_components + 1)]
    return pd.DataFrame(projected, columns=column_names).astype(FEATURE_DTYPE)

In [ ]:
def project_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    projection: dict[str, np.ndarray],
    n_components: int,
) -> str:
    """Project one Parquet file and write it to the target folder under the same name."""
    df = read_parquet_file(source_folder_name, file_name)
    projected_features = project_features(df, projection, n_components)
    return write_dataframe_to_parquet(projected_features, target_folder_name, file_name)

## 9.7. Projecting Every Split

*How are all three run?*

`process_all_splits` for the fourth time, with the arrays read once and handed to every worker.

`n_components` is passed in rather than defaulted, because it is computed in 9.5: a default would be
frozen at the moment this cell was executed, and re-running 9.5 with a different target would leave it
silently stale.

In [ ]:
def project_all_splits(
    source_folder_name: str,
    target_folder_name: str,
    n_components: int,
    projection_file: str = PATH_IPCA,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> None:
    """Project all three splits onto the components fitted on the training split."""
    projection = load_ipca(projection_file)
    process_all_splits(
        source_folder_name, target_folder_name, split_names, project_file, projection, n_components
    )

## 9.8. Run the Projection

Reads `data-scaled/` and fills `data-ipca/`, 504 files in and 504 out, each one 78 columns narrower by
54. This is the last shape the features take before the models: from here on a row is 24 numbers, and
its label is one integer in `data-encoded-label/`.

In [ ]:
project_all_splits(PATH_FOLDER_SCALED, PATH_FOLDER_IPCA, IPCA_COMPONENTS)

## 9.9. Check the Result

*Did the projection keep what the curve promised?*

The curve is a promise about variance made from the training split; the projected files are where it
either holds or does not. For each split the table measures how much variance the `pc` columns actually
carry and puts it beside `explained_variance`, the amount the fit predicted.

Three things are worth reading in it. The training row should match the prediction closely — that is
the projection doing what it was fitted to do. Dev and test should be near it but not equal, for the
fourth time in this notebook and for the same reason. And every split's means should sit at
approximately zero, because subtracting the training centre is part of the transformation.

In [ ]:
def summarize_projection(
    folder_name: str,
    projection_file: str = PATH_IPCA,
    split_names: tuple[str, ...] = SPLIT_NAMES,
) -> pd.DataFrame:
    """Variance carried by the projected columns of each split, beside what the fit predicted."""
    fitted_variance = load_ipca(projection_file)["explained_variance"]

    summary = {}
    for split_name in split_names:
        measured = measure_feature_statistics(folder_name, split_name)
        summary[split_name] = {
            "components": len(measured),
            "variance carried": (measured["std"] ** 2).sum(),
            "variance predicted": fitted_variance[: len(measured)].sum(),
            "largest |mean|": measured["mean"].abs().max(),
        }

    return pd.DataFrame(summary).T

In [ ]:
summarize_projection(PATH_FOLDER_IPCA)

The training row reads `66.87` carried against `66.87` predicted — the projection keeps what the fit
said it would, and `66.87` is `95.53%` of the 70 units of variance the scaled features hold, which is
the arithmetic of the whole section in one line. Its largest mean is $2.8 \times 10^{-9}$: centred, as
subtracting the training mean requires.

Dev carries **more** variance than train (`78.85`) and test **less** (`60.41`). Neither is a fault in
the projection — both were transformed by exactly the same 24 axes. It is the heavy tail of network
flow data showing through a random split: extreme rows are rare, so which split receives them is luck,
and 8.8 saw the same thing from the other side when one dev column came out 5.7 times wider than its
training counterpart. A model trained on the `train` rows will meet exactly this at prediction time,
which is the argument for measuring it rather than smoothing it away.

# 10. Synthetic Minority Oversampling Technique

The training split is nowhere near balanced. `Benign` accounts for 8,090,820 of its 9,739,760 rows —
**83%** — while the smallest class, `SQL Injection`, has **52**. A classifier that answers "benign" to
everything scores 83% on that split, and a model trained on it has very little reason to learn what 52
rows look like.

SMOTE (Synthetic Minority Over-sampling TEchnique) answers that by inventing rows rather than copying
them: it takes a minority row, picks one of its `k` nearest neighbours *of the same class*, and places
a new row somewhere on the line between the two. The additions are plausible blends of real rows rather
than duplicates, which a model cannot memorise the way it memorises the same row repeated a thousand
times.

**In:** `data-ipca/train/*.parquet` and `data-encoded-label/train/*.parquet` $\rightarrow$
**Out:** `data-smote/train/*.parquet`, features and label in the same file

Three things in that line are deliberate:

**The training split only.** Dev and test keep the real class balance. Resampling them would measure
the model against a world that does not exist, and every number it produced would be both flattering
and meaningless. It is the rule of 6.1 once more, applied to rows instead of statistics.

**The PCA features.** SMOTE interpolates between neighbours, so it needs a representation in which
distance means something: the 24 standardized, uncorrelated components of section 9 — which is also
exactly what the models will be given.

**Features and label in one file.** Everything until now kept the two in separate trees paired by row
order (5.4). Resampling breaks that pairing — the output has rows the label tree has no counterpart
for — so from here the label travels with the features.

### What per-file resampling actually does

The training split is 168 files, each a chunk of one day in capture order, and each is resampled on its
own. That has consequences which the code does not show:

- **123 of the 168 files hold a single class.** They fall outside their day's attack window, so they
  are entirely `Benign`; `fit_resample` needs at least two classes, and those files are passed through
  unchanged.
- The remaining **45 files are balanced to their own local majority**, not to any global target. In a
  2018-02-14 chunk holding 59,931 `FTP-BruteForce` rows and 69 `Benign` rows, the minority class is
  *`Benign`* — so SMOTE manufactures 59,862 synthetic **benign** rows in that file.

10.6 measures what this adds up to over the whole split. It is not a balanced training set, and the
gap between what the technique is for and what per-file application achieves is worth reading before
the results in the sections that follow.

### Why there is no fitted transformer here

SMOTE is a *sampler*, not a transformer: `fit_resample(X, y)` finds the neighbours of the rows it is
handed and returns new rows. Nothing carries from one call to the next — verified directly, calling
`fit(X, y)` before `fit_resample(X, y)` gives output identical to `fit_resample` alone.

The earlier pipeline fitted a `SMOTE` on the entire training split — 9.7 million rows in memory at once
— pickled it to `cache/smote/`, and then called `fit_resample` per file anyway, which redoes the work
from that file's own rows. The one thing that global fit contributed was a `k_neighbors` value, and it
came out as **5**: the default, because the clamp only bites when a class has fewer than 6 members and
the smallest class in the split has 52. So this section has no fit, no cached artefact and no 3 GB
load. It has one parameter and a per-file clamp that does the same job where a chunk is thin.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 10.1 Settings | What does SMOTE have to be told? | — |
| 10.2 Resampling the rows of one file | What happens to a file's rows? | 10.1 |
| 10.3 Resampling one file | What is read, and what is written? | 10.2 |
| 10.4 Resampling the training split | How are the 168 files run? | 10.1, 10.3 |
| 10.5 Run | — | 10.1, 10.4 |
| 10.6 Check the result | What did it actually change? | 10.1 |

## 10.1. Settings

Two inputs — `PATH_FOLDER_IPCA` (9.1) and `PATH_FOLDER_ENCODED_LABEL` (7.1) — one output, and one
parameter.

`SMOTE_K_NEIGHBORS` is how many same-class neighbours a row may be interpolated towards. Five is
imbalanced-learn's default and what the earlier pipeline used; a larger `k` blends more distant rows, a
smaller one keeps the synthetic rows closer to the originals.

`RANDOM_STATE` is the seed from 5.1 again. The new rows are random draws along those lines, so without
a fixed seed the training set would differ on every run and two models could not be compared.

`FIT_SPLIT` (6.1) names the only split that may be touched — for the last time in this notebook.

In [ ]:
PATH_FOLDER_SMOTE = "data-smote"

In [ ]:
SMOTE_K_NEIGHBORS = 5

## 10.2. Resampling the Rows of One File

*What happens to a file's rows?*

Two guards and one call.

A file with a single class is handed back untouched — 123 of the 168 are, and `fit_resample` raises
rather than returning them unchanged.

`choose_k_neighbors` is the second guard. SMOTE draws its line from a minority row to one of its `k`
nearest neighbours *of the same class*, so that class needs `k + 1` members before it has `k`
neighbours to offer. Across the whole split the smallest class has 52 rows and 5 is safe, but a single
chunk can be thinner: one training file holds a class with 5 rows, where `k` has to come down to 4.
Clamping turns that from an exception into a slightly less varied interpolation.

`fit_resample` accepts and returns pandas, so the column names survive the call and nothing has to be
rebuilt around a bare array. The original rows come back unchanged, with the synthetic ones appended
after them.

In [ ]:
def choose_k_neighbors(labels: pd.Series, k_neighbors: int = SMOTE_K_NEIGHBORS) -> int:
    """A class needs `k + 1` members to offer `k` neighbours, so a thin file gets a smaller `k`."""
    smallest_class_count = int(labels.value_counts().min())
    return max(1, min(k_neighbors, smallest_class_count - 1))

In [ ]:
def resample_rows(
    features: pd.DataFrame,
    labels: pd.Series,
    k_neighbors: int = SMOTE_K_NEIGHBORS,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.Series]:
    """Balance the classes of one file by interpolating new minority rows."""
    if labels.nunique() < 2:
        return features, labels

    smote = SMOTE(
        k_neighbors=choose_k_neighbors(labels, k_neighbors), random_state=random_state
    )
    resampled_features, resampled_labels = smote.fit_resample(features, labels)
    return resampled_features.astype(FEATURE_DTYPE), resampled_labels

## 10.3. Resampling One File

*What is read, and what is written?*

This is the only step in the notebook that reads from two trees at once. It can, because the file names
have matched across both trees since 5.4, and it must, because SMOTE cannot interpolate a row without
knowing its class.

What it writes is the two joined: 24 components and one label in a single frame. The label is assigned
through `to_numpy()` so the two are paired by position, which is what they are — two halves of the same
rows, in the same order — rather than by whatever indexes `fit_resample` handed back.

In [ ]:
def resample_file(
    features_folder_name: str,
    labels_folder_name: str,
    target_folder_name: str,
    file_name: str,
    k_neighbors: int = SMOTE_K_NEIGHBORS,
    random_state: int = RANDOM_STATE,
) -> str:
    """Resample one training file and write its features and label together under the same name."""
    features = read_parquet_file(features_folder_name, file_name)
    labels = read_parquet_file(labels_folder_name, file_name)[LABEL_COLUMN]

    resampled_features, resampled_labels = resample_rows(
        features, labels, k_neighbors, random_state
    )
    resampled = resampled_features.assign(**{LABEL_COLUMN: resampled_labels.to_numpy()})

    return write_dataframe_to_parquet(resampled, target_folder_name, file_name)

## 10.4. Resampling the Training Split

*How are the 168 files run?*

One task per file, as everywhere else, but not through `process_all_splits`: that walks three splits
and one source tree, and this walks one split and two. The difference is the whole point of the section
— only `train` is resampled — so it is spelled out here rather than hidden behind a parameter.

In [ ]:
def resample_training_split(
    features_folder_name: str,
    labels_folder_name: str,
    target_folder_name: str,
    split_name: str = FIT_SPLIT,
    k_neighbors: int = SMOTE_K_NEIGHBORS,
    random_state: int = RANDOM_STATE,
) -> None:
    """Resample every file of the training split, one worker process per file."""
    features_folder = get_split_folder(features_folder_name, split_name)
    labels_folder = get_split_folder(labels_folder_name, split_name)
    target_folder = get_split_folder(target_folder_name, split_name)
    create_split_folders(target_folder_name, (split_name,))

    file_names = get_all_file_names_in_folder(features_folder, "parquet")
    written_files = run_tasks_in_parallel(
        delayed(resample_file)(
            features_folder,
            labels_folder,
            target_folder,
            file_name,
            k_neighbors,
            random_state,
        )
        for file_name in file_names
    )

    print(f"Resampled {len(written_files)} {split_name} files into {target_folder}/.")

## 10.5. Run the Resampling

Reads `data-ipca/train/` and `data-encoded-label/train/` and fills `data-smote/train/`: 168 files in,
168 out, with more rows in them than went in. Dev and test are not named in this cell and never will
be.

In [ ]:
resample_training_split(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL, PATH_FOLDER_SMOTE)

## 10.6. Check the Result

*What did it actually change?*

Counting the classes before and after is the only way to see what per-file resampling amounts to, since
no single file shows it and the code cannot say it.

The chart is the same numbers in the shape they make. Five orders of magnitude separate the largest
class from the smallest, so the axis is logarithmic and the marks are dots rather than bars — a bar on
a log axis no longer encodes its value by its length. Where a class was left alone, its two dots sit on
top of each other.

In [ ]:
def compare_class_counts(
    labels_folder_name: str,
    resampled_folder_name: str,
    encoder_file: str = PATH_LABEL_ENCODER,
    split_name: str = FIT_SPLIT,
    label_column: str = LABEL_COLUMN,
) -> pd.DataFrame:
    """Rows per class before and after resampling, with the name each class id stands for."""
    before = count_labels_in_folder(
        get_split_folder(labels_folder_name, split_name), label_column
    )
    after = count_labels_in_folder(
        get_split_folder(resampled_folder_name, split_name), label_column
    )

    class_names = load_label_classes(encoder_file)
    table = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int).sort_index()
    table.insert(0, "class", [class_names[class_id] for class_id in table.index])
    table["synthetic"] = table["after"] - table["before"]
    table["share after %"] = (table["after"] / table["after"].sum() * 100).round(1)

    print(
        f"{table['before'].sum():,} rows in, {table['after'].sum():,} rows out, "
        f"{table['synthetic'].sum():,} of them synthetic."
    )
    return table

In [ ]:
def plot_class_counts(comparison: pd.DataFrame) -> None:
    """Draw the training rows of every class before and after resampling, on a logarithmic axis."""
    surface, grid, ink, muted = "#fcfcfb", "#e8e7e3", "#0b0b0b", "#52514e"
    before_colour, after_colour = "#2a78d6", "#eb6834"

    ordered = comparison.sort_values("before")
    positions = np.arange(len(ordered))

    figure, axes = plt.subplots(figsize=(9, 6))
    figure.patch.set_facecolor(surface)
    axes.set_facecolor(surface)

    axes.hlines(positions, ordered["before"], ordered["after"], color=grid, linewidth=2, zorder=1)
    for column, colour, label, layer in (
        ("before", before_colour, "before", 2),
        ("after", after_colour, "after", 3),
    ):
        axes.plot(
            ordered[column], positions, "o", linestyle="none", markersize=8, color=colour,
            markeredgecolor=surface, markeredgewidth=2, label=label, zorder=layer,
        )

    axes.set_xscale("log")
    axes.set_yticks(positions)
    axes.set_yticklabels(ordered["class"], fontsize=9)
    axes.set_xlabel("Training rows (log scale)", color=muted, fontsize=9)
    axes.set_title(
        "Training rows per class, before and after SMOTE", color=ink, fontsize=11, loc="left", pad=12
    )
    axes.grid(axis="x", color=grid, linewidth=1)
    axes.set_axisbelow(True)
    for side in ("top", "right", "left"):
        axes.spines[side].set_visible(False)
    axes.spines["bottom"].set_color(grid)
    axes.tick_params(colors=muted, labelsize=9, length=0)

    legend = axes.legend(loc="lower right", frameon=False, fontsize=9)
    for text in legend.get_texts():
        text.set_color(muted)

    figure.tight_layout()
    plt.show()

In [ ]:
class_counts = compare_class_counts(PATH_FOLDER_ENCODED_LABEL, PATH_FOLDER_SMOTE)
class_counts

In [ ]:
plot_class_counts(class_counts)

9,739,760 rows in, 11,365,674 out — 1,625,914 of them synthetic.

The rarest classes gain the most, which is what the technique is for. `SQL Injection` goes from 52 rows
to 119,443, so **99.96% of what a model sees of that class is interpolated from 52 real points**; the
same holds for `Brute Force -XSS` (138 rows) and `Brute Force -Web` (367). Whether a boundary learned
almost entirely from interpolation recognises real SQL injection traffic is a question only the test
split can answer — those three rows of the final classification report are worth reading before
trusting them.

The other half of the picture is the one the name does not suggest: **`Benign` gains 934,790 synthetic
rows**, 57% of everything created here, because in the 45 mixed files it is usually the *local*
minority. Three classes are left exactly as they were, and when it is over `Benign` is still 79% of the
training split against 83% before. Per-file resampling reduced the imbalance; it did not remove it.

If a genuinely balanced training set is the goal, the thing to revisit is the per-file granularity
rather than the code: resampling the split as a whole, or in stratified batches, with an explicit
`sampling_strategy` ratio does what the section title promises — at the cost of holding far more of the
split in memory at once, and of many more synthetic rows than 1.6 million. That is a change of method
rather than of style, so it is left here as a decision rather than made.

# 11. Load Datasets

Everything until now worked one file at a time: read a chunk, change it, write it, forget it. That
stops here. `KNeighborsClassifier`, `SVC`, `RandomForestClassifier` and `XGBClassifier` all want an
array, not a folder, so the three splits are read into memory once and the models in the sections that
follow are handed the same objects.

| variables | features from | labels from | rows |
| --- | --- | --- | --- |
| `train_x`, `train_y` | `data-smote/train` | the same files | 11,365,674 |
| `dev_x`, `dev_y` | `data-ipca/dev` | `data-encoded-label/dev` | 3,246,589 |
| `test_x`, `test_y` | `data-ipca/test` | `data-encoded-label/test` | 3,246,594 |

The training row is different from the other two in both columns, and both differences are the point
of the sections above. Its features come from `data-smote/` because it is the only split that was
resampled (10.5), and its label sits *inside* those files because resampling added rows that the label
tree has no counterpart for (10.3). Dev and test were never resampled, so they still arrive as two
trees paired by row order — the pairing set up in 5.4 and carried through every step since.

About 1.6 GB of features in total: roughly 1 GB of training rows and 297 MB each of dev and test, as
`float32`. Nothing is cast on the way in. 8.5 and 9.6 already wrote `float32`, and the labels are class
ids, which are integers — the earlier pipeline cast *every* numeric column to `float32` as it loaded,
which quietly turned class `11` into `11.0` in `train_y`, `dev_y` and `test_y` alike.

The three functions below are what the three parts share: read a split, pair two trees, and say what
was loaded.

In [ ]:
def load_split_frame(folder_name: str, split_name: str) -> pl.DataFrame:
    """Read every Parquet file of one split into a single frame in memory.

    The files are scanned in the order `get_split_file_paths` lists them — the same order for any
    folder, which is what keeps a feature tree and a label tree row-for-row comparable.
    """
    file_paths = get_split_file_paths(folder_name, split_name)
    return pl.scan_parquet(file_paths).collect(engine="streaming")

In [ ]:
def split_features_and_labels(
    frame: pl.DataFrame, label_column: str = LABEL_COLUMN
) -> tuple[pl.DataFrame, pl.Series]:
    """Separate the label column from a frame that carries both, as 10.3 wrote them."""
    return frame.drop(label_column), frame[label_column]

In [ ]:
def load_features_and_labels(
    features_folder_name: str,
    labels_folder_name: str,
    split_name: str,
    label_column: str = LABEL_COLUMN,
) -> tuple[pl.DataFrame, pl.Series]:
    """Load one split's features and labels from their two trees, paired by position."""
    features = load_split_frame(features_folder_name, split_name)
    labels = load_split_frame(labels_folder_name, split_name)[label_column]

    if features.height != labels.len():
        raise ValueError(
            f"{split_name}: {features.height:,} feature rows against {labels.len():,} labels — "
            "the two trees are no longer row-for-row."
        )
    return features, labels

In [ ]:
def describe_dataset(name: str, features: pl.DataFrame, labels: pl.Series) -> None:
    """Say what was loaded: how much of it, of what type, and how much memory it holds."""
    print(
        f"{name}: {features.height:,} rows x {features.width} columns "
        f"({features.dtypes[0]}, {features.estimated_size('mb'):.0f} MB) | "
        f"labels {labels.dtype} over {labels.n_unique()} classes"
    )

## 11.1. Load Train Sets

The resampled tree, and the only split that has one. Its features and its label live in the same file,
so nothing has to be paired here — the label is simply taken back out of the frame.

`FIT_SPLIT` names the split for the last time in the notebook. Everything fitted in sections 6 to 9 was
fitted on it, section 10 resampled it, and it is the only data the models are allowed to learn from.

These 11.4 million rows are not the 9.7 million the split started with: 1.6 million of them are SMOTE's
synthetic rows, and 934,790 of those are synthetic `Benign` (10.6). It is worth keeping that in mind
when a model reports how well it does on the training data.

In [ ]:
train_x, train_y = split_features_and_labels(load_split_frame(PATH_FOLDER_SMOTE, FIT_SPLIT))
describe_dataset("train", train_x, train_y)

## 11.2. Load Validation Sets

Validation data, from the two trees, with the real class balance: 83% `Benign`, 18 rows of
`SQL Injection`, exactly as the capture produced it. Nothing was resampled and nothing was fitted on
it.

This is what every hyperparameter search in the sections below scores against. Because it is used
repeatedly — once per candidate — it slowly stops being unseen data: choosing the settings that do best
on `dev` fits the choice to `dev`. That is the reason a third split exists at all.

In [ ]:
dev_x, dev_y = load_features_and_labels(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL, "dev")
describe_dataset("dev", dev_x, dev_y)

## 11.3. Load Test Sets

Test data, loaded the same way and from the same untouched representation.

Loading it is not looking at it. It belongs at the very end: fit on `train`, choose on `dev`, and
measure once on `test` when there is nothing left to decide. Every look that changes a decision spends
a little of what makes the final number worth reporting.

In [ ]:
test_x, test_y = load_features_and_labels(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL, "test")
describe_dataset("test", test_x, test_y)

Three splits in memory: 11,365,674 training rows the models may learn from, 3,246,589 validation rows
they may be compared on, and 3,246,594 test rows that stay closed until the end. Features are
`float32`, labels are integers between 0 and 14, and every feature frame has as many rows as its label
series — checked on the way in rather than assumed.

That is the end of the preprocessing pipeline. From `cse-cic-ids2018/*.csv` to here: Parquet (2.1),
infinities to `NaN` (4), a stratified split (5), medians (6), class ids (7), standardization (8), 24
components (9) and SMOTE on the training split alone (10).

# 12. Training and Evaluation Setup

Five classifiers are trained on the same data and compared: K-nearest neighbours, a kernel SVM, a
random forest, softmax regression and gradient boosting. They are chosen to span the space rather than
to be exhaustive — one lazy distance-based learner, one margin-based, two tree ensembles built on
opposite principles, and one linear baseline that says how much of the problem is easy.

Every one of the five sections below has the same six parts, so they can be read in any order and
compared line for line:

| Part | What it does |
| --- | --- |
| Settings | the grid to search and the numbers this model needs |
| Search | score every combination on the validation split |
| Run the search | run it, save the table, take the winner |
| Train | refit the winning combination and save it |
| Predict | run the model over dev and test, caching the predictions |
| Report | accuracy and macro scores, per-class results, confusion matrix |

### Four decisions they share

**Macro F1 selects the model, not accuracy.** The dev split keeps the real class balance, so 83% of it
is `Benign` and a model that answers "benign" to everything scores 83% accuracy while detecting
nothing. Macro F1 averages the per-class F1 scores with equal weight, so the 18 `SQL Injection` rows
count as much as the 649,000 benign ones. Accuracy is still reported — it is just not what chooses.

**Searches run on a subsample, the final fit does not.** A brute-force KNN query costs
`O(n_query × n_train)` and kernel SVM training is between `O(n²)` and `O(n³)`, so a single combination
over 11.4M training rows would take longer than this entire study. `stratified_subsample` keeps the
class proportions and a floor of `SEARCH_DEV_MIN_PER_CLASS` rows per class, so the rare attacks are
still there to be scored. The subsample decides *which* hyperparameters win; every number that gets
reported comes from a model refit on the full split and scored on the full split.

**Dev chooses, test reports.** Every search scores on dev. Test is predicted once, at the end, and
nothing about it feeds back into a choice.

**Predictions are cached to disk.** Predicting 3.2M rows takes minutes to hours per model, so each
chunk is written to `prediction-result/<model>/<split>/` and a rerun resumes instead of restarting.
The catch is the other side of the same coin: a cached chunk is never recomputed, so **after
retraining a model the matching prediction folder has to be deleted** or the old predictions will be
reported as the new model's.

## 12.1. Settings

Three output folders, one chunk size, and the two numbers that define what "best" means.

`SEARCH_DEV_MIN_PER_CLASS = 200` is the floor described above. `PREDICT_CHUNK_SIZE = 10_000` is small
enough that a chunk of predictions fits comfortably on the card for every model and large enough that
320 chunks cover a split.

`CLASS_NAMES` is read once from the artefact section 7 wrote, and is what turns a class id back into a
name in every report below.

In [ ]:
PATH_FOLDER_MODEL = "trained-model"
PATH_FOLDER_PREDICTION = "prediction-result"
PATH_FOLDER_SEARCH_RESULT = "hyperparameter-search"

In [ ]:
PREDICT_CHUNK_SIZE = 10_000

In [ ]:
SCORE_COLUMN = "f1_macro"
SEARCH_DEV_MIN_PER_CLASS = 200

In [ ]:
CLASS_NAMES = load_label_classes(PATH_LABEL_ENCODER)
print(f"{len(CLASS_NAMES)} classes: {CLASS_NAMES[0]} ... {CLASS_NAMES[-1]}")

# 13. K-Nearest Neighbours

KNN does not learn anything. It keeps the training rows, and to classify a new row it finds the `k`
closest of them and takes their majority vote. Everything that would be training time in another model
is paid at prediction time instead, which makes it the most expensive of the five to *use* and the
cheapest to fit.

That shape is exactly why it belongs in a study about robustness against disturbed input: a model with
no learned parameters cannot smooth over a perturbation, so its answer follows the geometry of the
24 components directly.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 13.1 Settings | What is searched, and over how much data? | 12.1 |
| 13.2 Searching without refitting | How are 42 combinations scored with 3 queries? | 13.1 |
| 13.3 Run the search | — | 13.1, 13.2 |
| 13.4 Train | — | 13.1, 13.3 |
| 13.5 Predict | — | 13.1, 13.4 |
| 13.6 Report | How good is it, and where does it fail? | 12.1, 13.5 |

## 13.1. Settings

The grid is `k × metric × weighting` = 7 × 3 × 2 = **42 combinations**.

`k` runs over odd values only: an even `k` can tie between two classes and the tie is broken
arbitrarily. The three metrics are the ones that mean something on standardized PCA components —
Manhattan and Euclidean measure distance along and across the axes, cosine ignores magnitude and
compares direction only. Weighting decides whether all `k` neighbours vote equally or a nearer one
counts for more.

`KNN_SEARCH_TRAIN_SAMPLES` is the index the search queries against, and `KNN_SEARCH_DEV_SAMPLES` is
how many dev rows are queried. Both are small because the cost is their product: 500,000 × 200,000
distance computations per metric is already the bulk of this section's runtime.

The file name carries the hyperparameters so that two models from different searches cannot overwrite
each other.

In [ ]:
KNN_SUBFOLDER = "KNN"

In [ ]:
list_of_n_neighbors = [1, 3, 5, 7, 9, 11, 13]
list_of_distance_metrics = ["manhattan", "euclidean", "cosine"]
list_of_weightings = ["uniform", "distance"]

In [ ]:
KNN_SEARCH_TRAIN_SAMPLES = 500_000
KNN_SEARCH_DEV_SAMPLES = 200_000
KNN_SEARCH_QUERY_CHUNK = 20_000

In [ ]:
def get_knn_name(n_neighbors: int, metric: str, weighting: str) -> str:
    """`k=11, manhattan, uniform` -> `knn-k-11-m-manhattan-w-uniform.pkl`."""
    return f"knn-k-{n_neighbors}-m-{metric}-w-{weighting}.pkl"

## 13.2. Searching Without Refitting

*How are 42 combinations scored with 3 queries?*

This is the one search in the notebook that does not use `run_search`, because it does not need to fit
42 times. Fitting a KNN only builds an index; all the cost is in the query. And for a fixed metric, one
query answers every combination:

- `kneighbors` returns the neighbours **sorted by distance**, so the `k` nearest for any smaller `k`
  are the leading columns of the same result — no new query needed for a different `k`.
- Both weightings are re-votes over those same columns.

So the loop is one query per *metric* — three — and 42 cheap re-votes over the results. Scoring the
whole grid the naive way would be 42 full queries.

`knn_vote` does the counting with `np.bincount` over a flattened `(row, class)` index rather than a
Python loop over 200,000 query rows, which is what makes a re-vote cost milliseconds. A neighbour at
distance zero is a special case: its `1/d` weight is infinite, and the convention is that an exact
match decides the vote by itself, so those rows fall back to counting only their exact matches.

`kneighbors_in_chunks` exists because a single 200,000-row query would allocate a
200,000 × 13 distance matrix on the card in one go. It also gives the only honest progress signal
available: a count of finished chunks and an ETA extrapolated from them.

In [ ]:
def knn_vote(
    neighbor_labels: np.ndarray,
    neighbor_distances: np.ndarray,
    n_classes: int,
    weighting: str,
) -> np.ndarray:
    """Majority vote over neighbours that have already been found.

    Both arrays are `(n_query, k)`: the class of each neighbour, and how far away it was.
    """
    n_query, n_neighbors = neighbor_labels.shape

    if weighting == "uniform":
        weights = np.ones((n_query, n_neighbors), dtype=np.float64)
    else:
        with np.errstate(divide="ignore"):
            weights = 1.0 / neighbor_distances.astype(np.float64)

        exact_match = ~np.isfinite(weights)
        rows_with_exact_match = exact_match.any(axis=1)
        if rows_with_exact_match.any():
            weights[rows_with_exact_match] = exact_match[rows_with_exact_match].astype(np.float64)

    row_offset = np.arange(n_query, dtype=np.int64)[:, None] * n_classes
    flat_index = (row_offset + neighbor_labels).ravel()
    scores = np.bincount(flat_index, weights=weights.ravel(), minlength=n_query * n_classes)

    return scores.reshape(n_query, n_classes).argmax(axis=1).astype(np.int32)

In [ ]:
def kneighbors_in_chunks(
    index, query: np.ndarray, n_neighbors: int, chunk_size: int
) -> tuple[np.ndarray, np.ndarray]:
    """Query the neighbour index a chunk of rows at a time; return distances and indices."""
    total = query.shape[0]
    n_chunks = math.ceil(total / chunk_size)
    distances, indices = [], []
    started = time.time()

    for number, start in enumerate(range(0, total, chunk_size), start=1):
        chunk_distances, chunk_indices = index.kneighbors(
            query[start : start + chunk_size], n_neighbors=n_neighbors
        )
        distances.append(to_numpy_2d(chunk_distances).astype(np.float32, copy=False))
        indices.append(to_numpy_2d(chunk_indices).astype(np.int64, copy=False))

        elapsed = time.time() - started
        progress_bar(
            number, n_chunks, f"neighbours (elapsed {elapsed:,.0f}s, eta {elapsed / number * (n_chunks - number):,.0f}s)"
        )

    return np.vstack(distances), np.vstack(indices)

In [ ]:
def search_knn(
    train_x,
    train_y,
    dev_x,
    dev_y,
    list_n_neighbors: list[int] = list_of_n_neighbors,
    list_metrics: list[str] = list_of_distance_metrics,
    list_weightings: list[str] = list_of_weightings,
    train_samples: int | None = KNN_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = KNN_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    chunk_size: int = KNN_SEARCH_QUERY_CHUNK,
    random_state: int = RANDOM_STATE,
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every k / metric / weighting combination with one neighbour query per metric."""
    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_classes = int(max(search_train_y.max(), search_dev_y.max())) + 1
    max_neighbors = max(list_n_neighbors)
    rows = []

    for metric in list_metrics:
        print(f"Fitting neighbour index (metric={metric})...")
        index = NearestNeighbors(n_neighbors=max_neighbors, metric=metric)
        index.fit(search_train_x)

        neighbor_distances, neighbor_indices = kneighbors_in_chunks(
            index, search_dev_x, max_neighbors, chunk_size
        )
        neighbor_labels = search_train_y[neighbor_indices]

        for n_neighbors in list_n_neighbors:
            for weighting in list_weightings:
                pred = knn_vote(
                    neighbor_labels[:, :n_neighbors],
                    neighbor_distances[:, :n_neighbors],
                    n_classes,
                    weighting,
                )
                scores = evaluate(pred=pred, true=search_dev_y).iloc[0].to_dict()
                rows.append(
                    {"n_neighbors": n_neighbors, "metric": metric, "weights": weighting, **scores}
                )
                print(
                    f"  k={n_neighbors:>3} metric={metric:<10} weights={weighting:<8}"
                    f" {score_column}={scores[score_column]:.4f}"
                )

        del index, neighbor_distances, neighbor_indices, neighbor_labels
        free_gpu_memory()

    return pd.DataFrame(rows).sort_values(score_column, ascending=False, ignore_index=True)

## 13.3. Run the Search

The table is saved before anything is trained: a grid that took an hour should not exist only in a
notebook variable.

The winning combination is then read back out of the table rather than typed into a constant. That is
deliberate — a hard-coded `k` that no longer matches the search is the easiest way for a thesis to
report a model it never selected. To override the search, assign the three names below by hand.

In [ ]:
knn_search_results = search_knn(train_x, train_y, dev_x, dev_y)
save_search_results(knn_search_results, "knn-search.csv", PATH_FOLDER_SEARCH_RESULT)
knn_search_results

In [ ]:
best_knn = get_best_hyperparameter(knn_search_results, SCORE_COLUMN)
KNN_N_NEIGHBORS = int(best_knn["n_neighbors"])
KNN_METRIC = str(best_knn["metric"])
KNN_WEIGHTING = str(best_knn["weights"])

## 13.4. Train

"Training" a KNN is building an index over the training rows — there is nothing to optimise. The whole
11.4M-row split is used, not the search subsample, because every row it keeps is a row it can find a
neighbour in.

The saved file is the training data in an index structure, so it is by far the largest of the five
models on disk.

In [ ]:
def train_knn(
    train_x,
    train_y,
    n_neighbors: int,
    metric: str,
    weighting: str,
    subfolder: str = KNN_SUBFOLDER,
    folder_name: str = PATH_FOLDER_MODEL,
) -> str:
    """Build the neighbour index over the full training split and save it."""
    print(f"Training KNN: k={n_neighbors} metric={metric} weights={weighting}")

    model = KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric, weights=weighting)
    with step(f"indexing {len(train_y):,} rows"):
        model.fit(to_features(train_x), to_labels(train_y))

    with step("saving model"):
        file_path = dump_trained_model(
            model, get_knn_name(n_neighbors, metric, weighting), subfolder, folder_name
        )

    print(f"Saved model to {file_path}.")
    return file_path

In [ ]:
train_knn(train_x, train_y, KNN_N_NEIGHBORS, KNN_METRIC, KNN_WEIGHTING)

## 13.5. Predict

Two runs of `predict_in_chunks`, one per split, each into its own cache folder. The only thing this
section contributes is the one line that turns a chunk of rows into class ids; the chunking, the
caching and the bar come from section 1.

This is the slow part of the KNN section and of the notebook: every one of the 3.2M dev rows is
compared against the 11.4M indexed training rows.

In [ ]:
def knn_predict(
    model_name: str,
    x,
    split_name: str,
    subfolder: str = KNN_SUBFOLDER,
    chunk_size: int = PREDICT_CHUNK_SIZE,
) -> np.ndarray:
    """Predict one split with a saved KNN model, caching each chunk."""
    with step(f"loading {model_name}"):
        model = load_trained_model(model_name, subfolder, PATH_FOLDER_MODEL)

    predictions = predict_in_chunks(
        lambda rows: to_numpy(model.predict(to_features(rows))),
        x,
        get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name),
        "knn",
        chunk_size,
    )

    del model
    free_gpu_memory()
    return predictions

In [ ]:
knn_model_name = get_knn_name(KNN_N_NEIGHBORS, KNN_METRIC, KNN_WEIGHTING)
knn_predict(knn_model_name, dev_x, "dev")

In [ ]:
knn_predict(knn_model_name, test_x, "test")

## 13.6. Report

Three views of the same predictions, each answering a different question.

`get_evaluation_results` gives the four headline numbers on both splits — dev beside test, because a
large gap between them is the sign that the search overfitted the validation split.

The classification report breaks the macro average back into its parts, which is where an intrusion
detector is actually judged: a recall of 0.2 on `SQL Injection` means four out of five injections went
through, however good the average looks.

The confusion matrix says *what* the mistakes were. Rows are the real class, columns are what the model
called it, so a benign row in the `DoS attacks-Hulk` column is a false alarm and the reverse is a
missed attack.

In [ ]:
knn_dev_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, KNN_SUBFOLDER), "dev")
knn_test_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, KNN_SUBFOLDER), "test")

pd.concat(
    [
        get_evaluation_results(knn_dev_folder, dev_y).assign(split="dev"),
        get_evaluation_results(knn_test_folder, test_y).assign(split="test"),
    ],
    ignore_index=True,
)

In [ ]:
knn_test_pred = load_prediction_chunks(knn_test_folder)
get_classification_report(pred=knn_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

In [ ]:
get_confusion_matrix(pred=knn_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

# 14. Support Vector Machine

An SVM looks for the boundary with the widest margin between classes, and with an RBF kernel that
boundary can curve. It is the one model here that does not scale to the data: training costs between
`O(n²)` and `O(n³)` in the number of rows and prediction costs `O(n_query × n_support_vectors)`, so
11.4M training rows are out of reach on any hardware this study has.

It is therefore fit on a **stratified subsample** of the training split, and that is a compute
constraint rather than a methodological choice — it should be reported as one. Two things keep the
comparison fair anyway: only *training* is subsampled, so its dev and test numbers come from the same
full splits as the other four models; and the subsample is drawn after SMOTE, so it inherits the
balance section 10 established.

On the GPU this section imports `cuml.svm.SVC` directly rather than going through the `cuml.accel`
proxy. The proxy wraps sklearn's `SVC`, and its GPU path refuses any `y` with more than two classes —
with 15 attack classes every fit would have fallen back to single-core libsvm without saying so.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 14.1 Settings | What is searched, and on how many rows? | 12.1 |
| 14.2 Building and scoring one SVM | What does one combination cost? | 14.1 |
| 14.3 Run the search | — | 14.1, 14.2 |
| 14.4 Train | — | 14.1, 14.3 |
| 14.5 Predict | — | 14.1, 14.4 |
| 14.6 Report | How good is it, and where does it fail? | 12.1, 14.5 |

## 14.1. Settings

`gamma` only affects the RBF kernel, so pairing it with `linear` would duplicate every `C` for no
benefit — `build_svm_grid` pins it there. That gives `3 (linear) + 6 (rbf) = 9` combinations.

`C` is how much the fit is penalised for a misclassified training row: small values accept mistakes in
exchange for a wider margin, large values bend the boundary to avoid them. `gamma` is how far the
influence of a single row reaches.

The two sample sizes differ by a factor of six: 50,000 rows to compare nine combinations, 300,000 for
the model that is kept. Even the second is a fortieth of the training split, and it is the largest fit
that finishes in reasonable time.

In [ ]:
SVM_SUBFOLDER = "SVM"

In [ ]:
list_of_kernels = ["rbf", "linear"]
list_of_c = [0.1, 1.0, 10.0]
list_of_gamma = ["scale", 0.1]

In [ ]:
SVM_SEARCH_TRAIN_SAMPLES = 50_000
SVM_SEARCH_DEV_SAMPLES = 100_000
SVM_TRAIN_SAMPLES = 300_000

In [ ]:
SVM_CACHE_SIZE = 2048
SVM_TOLERANCE = 1e-3
SVM_MAX_ITER = -1

In [ ]:
def get_svm_name(kernel: str, c: float, gamma) -> str:
    """`rbf, 1.0, scale` -> `svm-k-rbf-c-1.0-g-scale.pkl`."""
    return f"svm-k-{kernel}-c-{c}-g-{gamma}.pkl"

## 14.2. Building and Scoring One SVM

*What does one combination cost?*

`fit_and_score_svm` returns the scores **and** two costs, because with this model the cost is part of
the decision. `fit_seconds` says whether a combination is affordable at all, and `n_support` is the
number of training rows the boundary ended up leaning on — it sets the price of every prediction
afterwards, so a combination that wins by a thousandth of an F1 point while doubling the support set
is not the one to keep.

`support_vector_count` is wrapped in a `try` because cuML's multiclass wrapper does not always expose
it, and a missing diagnostic must not end a search.

cuML's GPU solver refuses some combinations outright — a high `C` on a hard pairwise sub-problem
raises `Working set has already been initialized!` from its C++ layer, reproducibly and with the card
nearly empty, so it is a solver limit rather than memory. `run_search` records those with their error
text and moves on, which is why the results table can contain rows with no scores. **A `NaN` row is a
combination that was never scored, not one that scored badly** — if the best model would have been in
that region, it is simply not in the table.

In [ ]:
def build_svm_grid(kernels: list[str], list_c: list[float], list_gamma: list) -> list[tuple]:
    """Every kernel/C/gamma combination worth fitting — `gamma` is dropped for the linear kernel."""
    combinations = []
    for kernel, c in itertools.product(kernels, list_c):
        if kernel == "linear":
            combinations.append((kernel, c, "scale"))
            continue
        combinations.extend((kernel, c, gamma) for gamma in list_gamma)
    return combinations

In [ ]:
def build_svm(
    kernel: str,
    c: float,
    gamma,
    cache_size: int = SVM_CACHE_SIZE,
    tolerance: float = SVM_TOLERANCE,
    max_iter: int = SVM_MAX_ITER,
) -> SVC:
    """Construct one SVM; the same arguments mean the same thing on both backends."""
    return SVC(
        C=float(c),
        kernel=kernel,
        gamma=gamma,
        cache_size=cache_size,
        tol=tolerance,
        max_iter=max_iter,
    )

In [ ]:
def support_vector_count(model) -> float:
    """How many training rows the boundary leans on, or `NaN` if the backend does not say."""
    try:
        n_support = getattr(model, "n_support_", None)
        if n_support is not None:
            return float(np.sum(to_numpy_2d(n_support)))

        support = getattr(model, "support_", None)
        if support is not None:
            return float(np.size(to_numpy_2d(support)))
    except Exception:
        pass
    return float("nan")

In [ ]:
def fit_and_score_svm(kernel, c, gamma, train_x, train_y, dev_x, dev_y) -> dict:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_svm(kernel, c, gamma)

    fit_started = time.time()
    with step(f"fitting on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y)
    fit_seconds = time.time() - fit_started

    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(dev_x))

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    measurements = {"fit_seconds": fit_seconds, "n_support": support_vector_count(model), **scores}

    del model
    free_gpu_memory()
    return measurements

## 14.3. Run the Search

`run_search` (section 1) walks the grid; what this section supplies is the grid, the names of its
three axes, the subsampled data and `fit_and_score_svm`. The same four arguments appear in the three
sections that follow, which is what makes those searches three lines each.

In [ ]:
def search_svm(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = SVM_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = SVM_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = RANDOM_STATE,
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every kernel/C/gamma combination on a subsample of dev."""
    if combinations is None:
        combinations = build_svm_grid(list_of_kernels, list_of_c, list_of_gamma)

    print(f"SVM search over {len(combinations)} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_data = (
        *stratified_subsample(train_x, train_y, train_samples, random_state),
        *stratified_subsample(dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class),
    )

    return run_search(
        combinations, ("kernel", "C", "gamma"), fit_and_score_svm, search_data, score_column
    )

In [ ]:
svm_search_results = search_svm(train_x, train_y, dev_x, dev_y)
save_search_results(svm_search_results, "svm-search.csv", PATH_FOLDER_SEARCH_RESULT)
svm_search_results

In [ ]:
best_svm = get_best_hyperparameter(svm_search_results, SCORE_COLUMN)
SVM_KERNEL = str(best_svm["kernel"])
SVM_C = float(best_svm["C"])
SVM_GAMMA = best_svm["gamma"]

## 14.4. Train

The one model in the notebook whose final fit is still a subsample — `SVM_TRAIN_SAMPLES` rows of the
11.4M available. The support-vector count is printed because it is the number that predicts how long
14.5 will take.

In [ ]:
def train_svm(
    train_x,
    train_y,
    kernel: str,
    c: float,
    gamma,
    train_samples: int | None = SVM_TRAIN_SAMPLES,
    random_state: int = RANDOM_STATE,
    subfolder: str = SVM_SUBFOLDER,
    folder_name: str = PATH_FOLDER_MODEL,
) -> str:
    """Fit the chosen SVM on a stratified subsample of the training split and save it."""
    print(f"Training SVM: kernel={kernel} C={c} gamma={gamma}")

    print("Preparing training subsample...")
    features, labels = stratified_subsample(train_x, train_y, train_samples, random_state)
    n_rows, n_features = features.shape

    model = build_svm(kernel, c, gamma)
    with step(f"fitting on {n_rows:,} rows x {n_features} features"):
        model.fit(features, labels)
    print(f"  {support_vector_count(model):,.0f} support vectors")

    with step("saving model"):
        file_path = dump_trained_model(
            model, get_svm_name(kernel, c, gamma), subfolder, folder_name
        )

    print(f"Saved model to {file_path}.")
    return file_path

In [ ]:
train_svm(train_x, train_y, SVM_KERNEL, SVM_C, SVM_GAMMA)

## 14.5. Predict

Identical in shape to 13.5, and for the same reason: only the line that turns rows into class ids
differs between models. The cost here is `n_query × n_support_vectors`, so the support-vector count
printed in 14.4 is what decides how long these two cells run.

In [ ]:
def svm_predict(
    model_name: str,
    x,
    split_name: str,
    subfolder: str = SVM_SUBFOLDER,
    chunk_size: int = PREDICT_CHUNK_SIZE,
) -> np.ndarray:
    """Predict one split with a saved SVM, caching each chunk."""
    with step(f"loading {model_name}"):
        model = load_trained_model(model_name, subfolder, PATH_FOLDER_MODEL)
    print(f"  {support_vector_count(model):,.0f} support vectors")

    predictions = predict_in_chunks(
        lambda rows: to_numpy(model.predict(to_features(rows))),
        x,
        get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name),
        "svm",
        chunk_size,
    )

    del model
    free_gpu_memory()
    return predictions

In [ ]:
svm_model_name = get_svm_name(SVM_KERNEL, SVM_C, SVM_GAMMA)
svm_predict(svm_model_name, dev_x, "dev")

In [ ]:
svm_predict(svm_model_name, test_x, "test")

## 14.6. Report

The same three views as 13.6. Read them knowing that this model saw 300,000 training rows against the
other four's 11.4 million: a weaker result here is not evidence that margins are the wrong idea for
this problem, only that a kernel SVM could not be given the data.

In [ ]:
svm_dev_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, SVM_SUBFOLDER), "dev")
svm_test_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, SVM_SUBFOLDER), "test")

pd.concat(
    [
        get_evaluation_results(svm_dev_folder, dev_y).assign(split="dev"),
        get_evaluation_results(svm_test_folder, test_y).assign(split="test"),
    ],
    ignore_index=True,
)

In [ ]:
svm_test_pred = load_prediction_chunks(svm_test_folder)
get_classification_report(pred=svm_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

In [ ]:
get_confusion_matrix(pred=svm_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

# 15. Random Forest

A forest grows a few hundred deep decision trees, each on a bootstrap sample of the rows and each
allowed to split on only a random handful of the components, then takes their majority vote. The trees
are individually overfitted and mutually decorrelated, and averaging them is what turns that into a
stable classifier.

It is the first model here that needs **no training subsample**. A tree costs about `O(n log n)` to
build and the trees are independent, so the full 11.4M-row split is affordable where the kernel SVM
was not. Only the search is subsampled, and only to keep twelve combinations down to minutes.

Two cuML details matter when reading the numbers. It splits on **quantiles** (`n_bins`, 128 here)
rather than on exact feature values, so its trees are not identical to a CPU forest with the same
settings; and `n_streams > 1` builds trees on several CUDA streams whose interleaving is not
deterministic, so `random_state` alone does not pin the forest exactly — set `RF_N_STREAMS = 1` if an
exactly reproducible forest matters more than speed. cuML also requires a finite `max_depth`, which is
why depth is an explicit search axis rather than something left to grow until the leaves are pure.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 15.1 Settings | What is searched? | 12.1 |
| 15.2 Building and scoring one forest | What does one combination cost? | 15.1 |
| 15.3 Run the search | — | 15.1, 15.2 |
| 15.4 Train | — | 15.1, 15.3 |
| 15.5 Predict | — | 15.1, 15.4 |
| 15.6 Report | How good is it, and where does it fail? | 12.1, 15.5 |

## 15.1. Settings

The grid is `n_estimators × max_depth × max_features` = 2 × 3 × 2 = **12 combinations**.

`max_features` is how many of the 24 components each split may choose from — `"sqrt"` gives about 5
and `"log2"` about 4.6 — and it is what decorrelates the trees from one another, which is the whole
reason the ensemble beats one deep tree. `max_depth` is the capacity knob, and `n_estimators` is how
many votes are averaged.

The search subsample is twenty times the SVM's, because forest training is near-linear in the rows
rather than quadratic, and a depth chosen on too few rows would not transfer to the full split.

In [ ]:
RF_SUBFOLDER = "RF"

In [ ]:
list_of_n_estimators = [100, 300]
list_of_max_depth = [12, 16, 24]
list_of_max_features = ["sqrt", "log2"]

In [ ]:
RF_SEARCH_TRAIN_SAMPLES = 1_000_000
RF_SEARCH_DEV_SAMPLES = 200_000
RF_TRAIN_SAMPLES = None

In [ ]:
RF_SPLIT_CRITERION = "gini"
RF_N_BINS = 128
RF_N_STREAMS = 4

In [ ]:
def get_rf_name(n_estimators: int, max_depth: int, max_features) -> str:
    """`300, 24, sqrt` -> `rf-n-300-d-24-f-sqrt.pkl`."""
    return f"rf-n-{n_estimators}-d-{max_depth}-f-{max_features}.pkl"

## 15.2. Building and Scoring One Forest

*What does one combination cost?*

`build_rf` is the only place in the notebook where the two backends disagree about names:
`split_criterion`, `n_bins` and `n_streams` are cuML's, sklearn calls the first `criterion` and has no
equivalent for the other two, so forwarding them unchanged would make the CPU path raise `TypeError`.
Keeping the fork inside one constructor is what lets everything above and below it ignore the
difference.

`fit_seconds` and `predict_seconds` are recorded beside the scores for the same reason `n_support` was
for the SVM. A forest of 300 deep trees can win the search by a hair and still cost several times more
per prediction than the runner-up, and predicting 3.2M rows is what that cost is paid on.

In [ ]:
def build_rf(
    n_estimators: int,
    max_depth: int,
    max_features,
    split_criterion: str = RF_SPLIT_CRITERION,
    n_bins: int = RF_N_BINS,
    n_streams: int = RF_N_STREAMS,
    random_state: int = RANDOM_STATE,
) -> RandomForestClassifier:
    """Construct the forest, translating the arguments the two backends name differently."""
    if use_cuda:
        return RandomForestClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth),
            max_features=max_features,
            split_criterion=split_criterion,
            n_bins=n_bins,
            n_streams=n_streams,
            random_state=random_state,
        )

    return RandomForestClassifier(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        max_features=max_features,
        criterion=split_criterion,
        random_state=random_state,
        n_jobs=-1,
    )

In [ ]:
def fit_and_score_rf(
    n_estimators, max_depth, max_features, train_x, train_y, dev_x, dev_y
) -> dict:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_rf(n_estimators, max_depth, max_features)

    fit_started = time.time()
    with step(f"fitting {n_estimators} trees on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y)
    fit_seconds = time.time() - fit_started

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(dev_x)).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    measurements = {
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **scores,
    }

    del model
    free_gpu_memory()
    return measurements

## 15.3. Run the Search

The same four lines as the SVM's, with a different grid and a different `fit_and_score`. The deepest,
largest forests are the ones most likely to exhaust the card, and `run_search` records a failure with
its error text and carries on — so check the `error` column before reading the table.

In [ ]:
def search_rf(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = RF_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = RF_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = RANDOM_STATE,
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every trees/depth/features combination on a subsample of dev."""
    if combinations is None:
        combinations = build_grid(list_of_n_estimators, list_of_max_depth, list_of_max_features)

    print(f"Random Forest search over {len(combinations)} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_data = (
        *stratified_subsample(train_x, train_y, train_samples, random_state),
        *stratified_subsample(dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class),
    )

    return run_search(
        combinations,
        ("n_estimators", "max_depth", "max_features"),
        fit_and_score_rf,
        search_data,
        score_column,
    )

In [ ]:
rf_search_results = search_rf(train_x, train_y, dev_x, dev_y)
save_search_results(rf_search_results, "rf-search.csv", PATH_FOLDER_SEARCH_RESULT)
rf_search_results

In [ ]:
best_rf = get_best_hyperparameter(rf_search_results, SCORE_COLUMN)
RF_N_ESTIMATORS = int(best_rf["n_estimators"])
RF_MAX_DEPTH = int(best_rf["max_depth"])
RF_MAX_FEATURES = best_rf["max_features"]

## 15.4. Train

Refit on the **full** training split: `RF_TRAIN_SAMPLES` is `None`, which `stratified_subsample` passes
straight through without dropping a row. Three hundred deep trees over 11.4M rows is the heaviest
single fit in the notebook.

If the card runs out of memory, either set `RF_N_STREAMS = 1` so fewer trees are built at once, or pass
a row budget — `train_rf(..., train_samples=5_000_000)` — to fit on a class-proportional subsample
instead. Both fits are saved under the same name, so a subsampled forest overwrites a full one with the
same hyperparameters.

In [ ]:
def train_rf(
    train_x,
    train_y,
    n_estimators: int,
    max_depth: int,
    max_features,
    train_samples: int | None = RF_TRAIN_SAMPLES,
    random_state: int = RANDOM_STATE,
    subfolder: str = RF_SUBFOLDER,
    folder_name: str = PATH_FOLDER_MODEL,
) -> str:
    """Fit the chosen forest on the training split and save it."""
    print(
        f"Training Random Forest: n_estimators={n_estimators} "
        f"max_depth={max_depth} max_features={max_features}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(train_x, train_y, train_samples, random_state)
    n_rows, n_features = features.shape

    model = build_rf(n_estimators, max_depth, max_features, random_state=random_state)
    with step(f"fitting {n_estimators} trees on {n_rows:,} rows x {n_features} features"):
        model.fit(features, labels)

    with step("saving model"):
        file_path = dump_trained_model(
            model, get_rf_name(n_estimators, max_depth, max_features), subfolder, folder_name
        )

    print(f"Saved model to {file_path}.")
    return file_path

In [ ]:
train_rf(train_x, train_y, RF_N_ESTIMATORS, RF_MAX_DEPTH, RF_MAX_FEATURES)

## 15.5. Predict

A forest predicts by walking every row down 300 trees and counting the votes, which is far cheaper per
row than the KNN's search through 11.4M neighbours or the SVM's comparison against its support set.

In [ ]:
def rf_predict(
    model_name: str,
    x,
    split_name: str,
    subfolder: str = RF_SUBFOLDER,
    chunk_size: int = PREDICT_CHUNK_SIZE,
) -> np.ndarray:
    """Predict one split with a saved forest, caching each chunk."""
    with step(f"loading {model_name}"):
        model = load_trained_model(model_name, subfolder, PATH_FOLDER_MODEL)

    predictions = predict_in_chunks(
        lambda rows: to_numpy(model.predict(to_features(rows))),
        x,
        get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name),
        "rf",
        chunk_size,
    )

    del model
    free_gpu_memory()
    return predictions

In [ ]:
rf_model_name = get_rf_name(RF_N_ESTIMATORS, RF_MAX_DEPTH, RF_MAX_FEATURES)
rf_predict(rf_model_name, dev_x, "dev")

In [ ]:
rf_predict(rf_model_name, test_x, "test")

## 15.6. Report

The same three views. This model saw every training row, so unlike the SVM its numbers can be read as
what a random forest does on this problem rather than as what one could manage on a fortieth of it.

In [ ]:
rf_dev_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, RF_SUBFOLDER), "dev")
rf_test_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, RF_SUBFOLDER), "test")

pd.concat(
    [
        get_evaluation_results(rf_dev_folder, dev_y).assign(split="dev"),
        get_evaluation_results(rf_test_folder, test_y).assign(split="test"),
    ],
    ignore_index=True,
)

In [ ]:
rf_test_pred = load_prediction_chunks(rf_test_folder)
get_classification_report(pred=rf_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

In [ ]:
get_confusion_matrix(pred=rf_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

# 16. Logistic Regression (Softmax)

The linear baseline of the study, and the only model here that cannot represent an interaction between
two features. It is a single `Dense(n_classes, activation="softmax")` layer applied straight to the 24
components — **no hidden layer and nothing non-linear in between** — so what is fitted is exactly
multinomial logistic regression,

$$P(y = k \mid x) = \mathrm{softmax}(Wx + b)_k$$

with $W$ of shape 15 × 24 and $b$ of length 15: **375 parameters**, against a forest of 300 trees.
Keras is used here as the optimiser and the GPU runtime, not as a way to build a deep model; a model
with no hidden representation is precisely what separates this baseline from the neural networks this
thesis is not about.

Keras rather than `sklearn.linear_model.LogisticRegression` or `cuml.linear_model` because those
solvers are full-batch: `lbfgs` holds the design matrix and takes a pass over all 11.4M rows per
iteration. `model.fit` streams mini-batches to the card instead, which is what makes the full split
affordable the same way it was for the forest.

The objective is convex, so the optimiser decides only how fast the single global optimum is reached,
not which model is found. That is why the search varies *learning rate*, *batch size* and *L2 strength*
rather than an architecture: the first two govern convergence and the third is the only capacity knob a
linear model has.

Two things carry over from the preprocessing chapter. Labels are integer class ids, so the loss is
`sparse_categorical_crossentropy` and no one-hot expansion of 11.4M × 15 is ever materialised. And no
`class_weight` is passed — section 10 already resampled the training split, and weighting it again
would correct the same imbalance twice.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 16.1 Settings | What is searched? | 12.1 |
| 16.2 Building and scoring one model | Why is `epochs` not an axis? | 16.1 |
| 16.3 Run the search | — | 16.1, 16.2 |
| 16.4 Train | — | 16.1, 16.3 |
| 16.5 Predict | — | 16.1, 16.4 |
| 16.6 Report | How good is it, and where does it fail? | 12.1, 16.5 |

## 16.1. Settings

The grid is `learning_rate × batch_size × l2` = 2 × 2 × 3 = **12 combinations**, the same size as the
forest's.

`LOGREG_MAX_EPOCHS` and `LOGREG_PATIENCE` are the early-stopping budget rather than search axes — see
16.2. `LOGREG_PREDICT_BATCH` is how many rows are pushed through the card at once when predicting,
which has nothing to do with the batch size used for fitting.

`LOGREG_N_CLASSES` comes from the class list section 7 wrote, so the output layer cannot disagree with
the encoding.

In [ ]:
LOGREG_SUBFOLDER = "LOGREG"

In [ ]:
list_of_learning_rates = [1e-2, 1e-3]
list_of_batch_sizes = [2048, 8192]
list_of_l2 = [0.0, 1e-5, 1e-4]

In [ ]:
LOGREG_SEARCH_TRAIN_SAMPLES = 1_000_000
LOGREG_SEARCH_DEV_SAMPLES = 200_000
LOGREG_TRAIN_SAMPLES = None

In [ ]:
LOGREG_N_CLASSES = len(CLASS_NAMES)
LOGREG_MAX_EPOCHS = 30
LOGREG_PATIENCE = 3
LOGREG_PREDICT_BATCH = 8192

In [ ]:
def get_logreg_name(learning_rate: float, batch_size: int, l2: float) -> str:
    """Keras models are not picklable, so this section saves `.keras` archives."""
    return f"logreg-lr-{learning_rate}-bs-{batch_size}-l2-{l2}.keras"

## 16.2. Building and Scoring One Model

*Why is `epochs` not a search axis?*

Because it is not a property of the model — it is how long the optimiser was allowed to run. Searching
over it would mostly rediscover that a smaller learning rate needs more epochs. Each combination is
instead given up to `LOGREG_MAX_EPOCHS` under an `EarlyStopping` callback watching the dev loss, and
the epoch count it actually stopped at is recorded as `epochs_run`; the winner's `epochs_run` is what
the final fit in 16.4 is given.

`restore_best_weights=True` matters more than it looks: without it a combination that began to overfit
would be scored on weights nobody would deploy.

One caveat to state plainly. The dev subsample is what early stopping monitors *and* what the
combination is scored on, so the search scores are mildly optimistic and should not be quoted as the
model's performance — the numbers in 16.6 come from a model refit without early stopping.

`keras.backend.clear_session()` joins `free_gpu_memory` here: TensorFlow keeps the graph and the
optimiser's slot variables alive per session, so `del model` alone leaks a little device memory on
every one of the twelve fits.

In [ ]:
def build_logreg(
    n_features: int,
    learning_rate: float,
    l2: float,
    n_classes: int = LOGREG_N_CLASSES,
    random_state: int = RANDOM_STATE,
) -> keras.Model:
    """One softmax layer over the input features — no hidden layer, nothing in between.

    `set_random_seed` seeds Python, NumPy and TensorFlow together, which is what makes the weight
    initialisation and the per-epoch shuffling repeatable. Unlike the forest there is no stream
    caveat: once the seed is fixed the fit is deterministic.
    """
    keras.utils.set_random_seed(random_state)

    model = keras.Sequential(
        [
            keras.Input(shape=(n_features,)),
            keras.layers.Dense(
                n_classes,
                activation="softmax",
                kernel_regularizer=keras.regularizers.L2(l2) if l2 else None,
                name="softmax",
            ),
        ],
        name="logistic_regression",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=float(learning_rate)),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
def fit_and_score_logreg(
    learning_rate,
    batch_size,
    l2,
    train_x,
    train_y,
    dev_x,
    dev_y,
    max_epochs: int = LOGREG_MAX_EPOCHS,
    patience: int = LOGREG_PATIENCE,
) -> dict:
    """Fit one combination under early stopping, score it on dev, and release the session."""
    model = build_logreg(train_x.shape[1], learning_rate, l2)

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True
    )

    fit_started = time.time()
    with step(f"fitting {train_x.shape[0]:,} rows for up to {max_epochs} epochs"):
        history = model.fit(
            train_x,
            train_y,
            validation_data=(dev_x, dev_y),
            epochs=max_epochs,
            batch_size=int(batch_size),
            callbacks=[early_stopping],
            shuffle=True,
            verbose=0,
        )
    fit_seconds = time.time() - fit_started

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        probabilities = model.predict(dev_x, batch_size=LOGREG_PREDICT_BATCH, verbose=0)
        pred = probabilities.argmax(axis=1).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    measurements = {
        "epochs_run": len(history.history["loss"]),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **scores,
    }

    del model
    keras.backend.clear_session()
    free_gpu_memory()
    return measurements

## 16.3. Run the Search

The same shape again. `epochs_run` comes back in the results table beside the scores, and the winning
row's value is what 16.4 trains for.

In [ ]:
def search_logreg(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = LOGREG_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = LOGREG_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = RANDOM_STATE,
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every learning rate / batch size / L2 combination on a subsample of dev."""
    if combinations is None:
        combinations = build_grid(list_of_learning_rates, list_of_batch_sizes, list_of_l2)

    print(f"Logistic Regression search over {len(combinations)} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_data = (
        *stratified_subsample(train_x, train_y, train_samples, random_state),
        *stratified_subsample(dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class),
    )

    return run_search(
        combinations,
        ("learning_rate", "batch_size", "l2"),
        fit_and_score_logreg,
        search_data,
        score_column,
    )

In [ ]:
logreg_search_results = search_logreg(train_x, train_y, dev_x, dev_y)
save_search_results(logreg_search_results, "logreg-search.csv", PATH_FOLDER_SEARCH_RESULT)
logreg_search_results

In [ ]:
best_logreg = get_best_hyperparameter(logreg_search_results, SCORE_COLUMN)
LOGREG_LEARNING_RATE = float(best_logreg["learning_rate"])
LOGREG_BATCH_SIZE = int(best_logreg["batch_size"])
LOGREG_L2 = float(best_logreg["l2"])
LOGREG_EPOCHS = int(best_logreg["epochs_run"])

## 16.4. Train

Refit on the full training split for a **fixed** number of epochs — the winner's `epochs_run` — with no
early stopping. That is what keeps the dev split clean: if the final fit stopped on a dev signal, dev
would have shaped the weights it is later used to judge.

At 375 parameters this is the cheapest fit in the notebook; the cost is moving 11.4M × 24 float32
values through the card, not the arithmetic. `verbose=2` prints one line per epoch instead of a bar
over several thousand steps.

In [ ]:
def dump_keras_model(model, name: str, subfolder: str, folder_name: str) -> str:
    """`dump_trained_model` for Keras: a compiled graph cannot be pickled, so it is saved natively."""
    target_folder = os.path.join(folder_name, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    model.save(file_path)
    return file_path

In [ ]:
def train_logreg(
    train_x,
    train_y,
    learning_rate: float,
    batch_size: int,
    l2: float,
    epochs: int,
    train_samples: int | None = LOGREG_TRAIN_SAMPLES,
    random_state: int = RANDOM_STATE,
    subfolder: str = LOGREG_SUBFOLDER,
    folder_name: str = PATH_FOLDER_MODEL,
) -> str:
    """Fit the chosen softmax regression on the training split for a fixed number of epochs."""
    print(
        f"Training Logistic Regression: learning_rate={learning_rate} "
        f"batch_size={batch_size} l2={l2} epochs={epochs}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(train_x, train_y, train_samples, random_state)
    n_rows, n_features = features.shape

    model = build_logreg(n_features, learning_rate, l2, random_state=random_state)
    model.summary()

    print(f"Fitting {epochs} epochs on {n_rows:,} rows x {n_features} features")
    model.fit(features, labels, epochs=int(epochs), batch_size=int(batch_size), shuffle=True, verbose=2)

    with step("saving model"):
        file_path = dump_keras_model(
            model, get_logreg_name(learning_rate, batch_size, l2), subfolder, folder_name
        )

    print(f"Saved model to {file_path}.")
    return file_path

In [ ]:
train_logreg(train_x, train_y, LOGREG_LEARNING_RATE, LOGREG_BATCH_SIZE, LOGREG_L2, LOGREG_EPOCHS)

## 16.5. Predict

`predict` returns a `(chunk, 15)` matrix of class probabilities; only the `argmax` is cached, so these
files stay comparable with the other four models' and the cache stays small. The probabilities
themselves are not kept — nothing downstream asks for a confidence.

In [ ]:
def logreg_predict(
    model_name: str,
    x,
    split_name: str,
    subfolder: str = LOGREG_SUBFOLDER,
    chunk_size: int = PREDICT_CHUNK_SIZE,
    batch_size: int = LOGREG_PREDICT_BATCH,
) -> np.ndarray:
    """Predict one split with a saved softmax model, caching each chunk."""
    with step(f"loading {model_name}"):
        model = keras.models.load_model(
            os.path.join(PATH_FOLDER_MODEL, subfolder, model_name)
        )

    predictions = predict_in_chunks(
        lambda rows: model.predict(
            to_features(rows), batch_size=batch_size, verbose=0
        ).argmax(axis=1),
        x,
        get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name),
        "logreg",
        chunk_size,
    )

    del model
    keras.backend.clear_session()
    free_gpu_memory()
    return predictions

In [ ]:
logreg_model_name = get_logreg_name(LOGREG_LEARNING_RATE, LOGREG_BATCH_SIZE, LOGREG_L2)
logreg_predict(logreg_model_name, dev_x, "dev")

In [ ]:
logreg_predict(logreg_model_name, test_x, "test")

## 16.6. Report

Read this one as the floor. Whatever the four non-linear models achieve above these numbers is what
their extra capacity bought; whatever they do not is a sign that the 24 components already separate the
classes linearly.

In [ ]:
logreg_dev_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, LOGREG_SUBFOLDER), "dev")
logreg_test_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, LOGREG_SUBFOLDER), "test")

pd.concat(
    [
        get_evaluation_results(logreg_dev_folder, dev_y).assign(split="dev"),
        get_evaluation_results(logreg_test_folder, test_y).assign(split="test"),
    ],
    ignore_index=True,
)

In [ ]:
logreg_test_pred = load_prediction_chunks(logreg_test_folder)
get_classification_report(pred=logreg_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

In [ ]:
get_confusion_matrix(pred=logreg_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

# 17. XGBoost

Gradient boosting is the counterpart to the random forest: both are ensembles of decision trees, but a
forest averages a few hundred *independent* deep trees while a booster fits shallow trees
*sequentially*, each one on the error the ones before it left behind.

That is why the two sections search different axes. A forest's error is dominated by how decorrelated
its trees are, which `max_features` controls; a booster's is dominated by how large a step each tree is
allowed to take (`learning_rate`) and how many steps it gets.

cuML has no boosting implementation, so this section uses XGBoost's own `XGBClassifier`. The card is
selected with `device="cuda"` and `tree_method="hist"` — the histogram builder is the only one with a
CUDA implementation, and asking for `"exact"` on the GPU silently falls back to the CPU. Histogram
building is the same quantile bucketing `RF_N_BINS` controls for the forest, so `XGB_MAX_BIN` is set to
the same 128 buckets and the two stay comparable.

This is the most device-memory-hungry fit per row in the notebook: a 15-class objective keeps a
gradient and a hessian **per class per row**.

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 17.1 Settings | What is searched? | 12.1 |
| 17.2 Building and scoring one booster | Why is `n_estimators` not an axis? | 17.1 |
| 17.3 Run the search | — | 17.1, 17.2 |
| 17.4 Train | — | 17.1, 17.3 |
| 17.5 Predict | — | 17.1, 17.4 |
| 17.6 Report | How good is it, and where does it fail? | 12.1, 17.5 |

## 17.1. Settings

The grid is `max_depth × learning_rate × colsample_bytree` = 3 × 2 × 2 = **12 combinations**.

`max_depth` runs 6/8/10 rather than the forest's 12/16/24: a boosted tree is a correction rather than a
standalone predictor, so it is conventionally much shallower, and depth is the axis that costs the most
— the histogram builder's work per round roughly doubles per level.

`colsample_bytree` is the boosting analogue of `max_features`: the fraction of the 24 components each
tree may split on. `0.6` gives about 14 of them and `1.0` all 24, so the axis measures whether
decorrelating the trees still buys anything once they are fitted sequentially.

The names are prefixed `list_of_xgb_*` rather than reusing the bare names from section 15 — `search_rf`
reads those as default arguments, and shadowing them here would silently change the forest's search on
a later rerun.

In [ ]:
XGB_SUBFOLDER = "XGB"

In [ ]:
list_of_xgb_max_depth = [6, 8, 10]
list_of_xgb_learning_rates = [0.1, 0.3]
list_of_xgb_colsample_bytree = [0.6, 1.0]

In [ ]:
XGB_SEARCH_TRAIN_SAMPLES = 1_000_000
XGB_SEARCH_DEV_SAMPLES = 200_000
XGB_TRAIN_SAMPLES = None

In [ ]:
XGB_DEVICE = "cuda" if use_cuda else "cpu"
XGB_TREE_METHOD = "hist"
XGB_MAX_ROUNDS = 400
XGB_EARLY_STOPPING_ROUNDS = 20
XGB_MAX_BIN = 128
XGB_SUBSAMPLE = 0.8
XGB_MIN_CHILD_WEIGHT = 1.0

In [ ]:
def get_xgb_name(max_depth: int, learning_rate: float, colsample_bytree: float) -> str:
    """Boosters are saved as native `.json` archives, which survive an XGBoost upgrade."""
    return f"xgb-d-{max_depth}-lr-{learning_rate}-c-{colsample_bytree}.json"

## 17.2. Building and Scoring One Booster

*Why is `n_estimators` not a search axis?*

For the same reason `epochs` was not one for the softmax regression: it is how long the fit was allowed
to run, not a property of the model, and it trades off directly against `learning_rate` — a grid over
both would mostly rediscover that a smaller step needs more steps. Each combination gets a budget of
`XGB_MAX_ROUNDS` with `early_stopping_rounds` watching dev `mlogloss`, and the round it stopped
improving at is recorded as `n_trees_used` for 17.4 to use.

`early_stopping_rounds` is a *constructor* argument in the sklearn wrapper — it moved out of `fit` in
XGBoost 1.6 — so the search passes it here and the final refit leaves it at `None`.

`to_xgb_features` is a small but necessary detail. `predict` has a fast path that needs its input where
the model lives; handing a NumPy array to a `device="cuda"` booster still returns the right answer, but
it rebuilds a `DMatrix` and warns about the device mismatch on *every* call — once per 10,000-row
chunk, so about 320 times per split. `fit` is left on host arrays, where XGBoost does the transfer
itself while binning.

One constraint is worth knowing about because it is XGBoost's alone: its scikit-learn wrapper requires
the class ids it is given to be `0 … n_classes - 1` with no gaps, and raises `Invalid classes inferred
from unique values of y` otherwise. The training split has all fifteen, so this never fires here — but
a subsample that happened to miss a class, or a grid run on a subset of the data, would stop this
model while the other four carried on.


In [ ]:
def build_xgb(
    max_depth: int,
    learning_rate: float,
    colsample_bytree: float,
    n_estimators: int = XGB_MAX_ROUNDS,
    early_stopping_rounds: int | None = None,
    subsample: float = XGB_SUBSAMPLE,
    min_child_weight: float = XGB_MIN_CHILD_WEIGHT,
    max_bin: int = XGB_MAX_BIN,
    tree_method: str = XGB_TREE_METHOD,
    device: str = XGB_DEVICE,
    random_state: int = RANDOM_STATE,
) -> XGBClassifier:
    """Construct the booster; the same class serves both devices."""
    return XGBClassifier(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        learning_rate=float(learning_rate),
        colsample_bytree=float(colsample_bytree),
        subsample=float(subsample),
        min_child_weight=float(min_child_weight),
        max_bin=int(max_bin),
        tree_method=tree_method,
        device=device,
        objective="multi:softprob",
        eval_metric="mlogloss",
        early_stopping_rounds=early_stopping_rounds,
        random_state=random_state,
    )

In [ ]:
def to_xgb_features(x):
    """The features as a `float32` matrix on whichever device the booster is on."""
    features = to_features(x)
    if not use_cuda:
        return features

    import cupy as cp

    return cp.asarray(features)

In [ ]:
def fit_and_score_xgb(
    max_depth,
    learning_rate,
    colsample_bytree,
    train_x,
    train_y,
    dev_x,
    dev_y,
    max_rounds: int = XGB_MAX_ROUNDS,
    early_stopping_rounds: int = XGB_EARLY_STOPPING_ROUNDS,
) -> dict:
    """Fit one combination under early stopping, score it on dev, and release the card."""
    model = build_xgb(
        max_depth,
        learning_rate,
        colsample_bytree,
        n_estimators=max_rounds,
        early_stopping_rounds=early_stopping_rounds,
    )

    fit_started = time.time()
    with step(f"boosting up to {max_rounds} rounds on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y, eval_set=[(dev_x, dev_y)], verbose=False)
    fit_seconds = time.time() - fit_started

    # `best_iteration` is a 0-based round index, so the tree count is one more than it.
    best_iteration = getattr(model, "best_iteration", None)
    n_trees_used = max_rounds if best_iteration is None else int(best_iteration) + 1

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(to_xgb_features(dev_x))).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    measurements = {
        "n_trees_used": n_trees_used,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **scores,
    }

    del model
    free_gpu_memory()
    return measurements

## 17.3. Run the Search

`free_gpu_memory` earns its place here: `cuml.accel` does not intercept XGBoost, so what releases a
finished booster is the `gc.collect()` inside that helper rather than the CuPy pool drain. It is still
called at the same points, so whatever cuML and TensorFlow are holding comes back between combinations.

In [ ]:
def search_xgb(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = XGB_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = XGB_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = RANDOM_STATE,
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every depth / learning rate / colsample combination on a subsample of dev."""
    if combinations is None:
        combinations = build_grid(
            list_of_xgb_max_depth, list_of_xgb_learning_rates, list_of_xgb_colsample_bytree
        )

    print(f"XGBoost search over {len(combinations)} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_data = (
        *stratified_subsample(train_x, train_y, train_samples, random_state),
        *stratified_subsample(dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class),
    )

    return run_search(
        combinations,
        ("max_depth", "learning_rate", "colsample_bytree"),
        fit_and_score_xgb,
        search_data,
        score_column,
    )

In [ ]:
xgb_search_results = search_xgb(train_x, train_y, dev_x, dev_y)
save_search_results(xgb_search_results, "xgb-search.csv", PATH_FOLDER_SEARCH_RESULT)
xgb_search_results

In [ ]:
best_xgb = get_best_hyperparameter(xgb_search_results, SCORE_COLUMN)
XGB_MAX_DEPTH = int(best_xgb["max_depth"])
XGB_LEARNING_RATE = float(best_xgb["learning_rate"])
XGB_COLSAMPLE_BYTREE = float(best_xgb["colsample_bytree"])
XGB_N_ESTIMATORS = int(best_xgb["n_trees_used"])

## 17.4. Train

Refit on the full training split for a fixed number of rounds, early stopping switched off — the same
arrangement the softmax regression uses for its epochs, and for the same reason: no part of dev may
shape the final trees.

This is the fit most likely to exhaust an 8 GiB card. At 11.4M rows the per-class gradients and
hessians alone are well over a gigabyte, on top of the cached margins and the binned feature matrix. If
it runs out, pass a row budget — `train_xgb(..., train_samples=5_000_000)` — rather than cutting the
tree count, which would change the model the search selected. Dropping `XGB_MAX_BIN` to 64 is the other
lever, at the cost of no longer matching the forest's bin count.

In [ ]:
def dump_xgb_model(model, name: str, subfolder: str, folder_name: str) -> str:
    """`dump_trained_model` for XGBoost: `save_model` writes a native archive rather than a pickle."""
    target_folder = os.path.join(folder_name, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    model.save_model(file_path)
    return file_path

In [ ]:
def train_xgb(
    train_x,
    train_y,
    max_depth: int,
    learning_rate: float,
    colsample_bytree: float,
    n_estimators: int,
    train_samples: int | None = XGB_TRAIN_SAMPLES,
    random_state: int = RANDOM_STATE,
    subfolder: str = XGB_SUBFOLDER,
    folder_name: str = PATH_FOLDER_MODEL,
) -> str:
    """Fit the chosen booster on the training split for a fixed number of rounds."""
    print(
        f"Training XGBoost: max_depth={max_depth} learning_rate={learning_rate} "
        f"colsample_bytree={colsample_bytree} n_estimators={n_estimators}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(train_x, train_y, train_samples, random_state)
    n_rows, n_features = features.shape

    model = build_xgb(
        max_depth, learning_rate, colsample_bytree, n_estimators=n_estimators, random_state=random_state
    )
    with step(f"boosting {n_estimators} rounds on {n_rows:,} rows x {n_features} features ({XGB_DEVICE})"):
        model.fit(features, labels, verbose=False)

    with step("saving model"):
        file_path = dump_xgb_model(
            model, get_xgb_name(max_depth, learning_rate, colsample_bytree), subfolder, folder_name
        )

    print(f"Saved model to {file_path}.")
    return file_path

In [ ]:
train_xgb(
    train_x, train_y, XGB_MAX_DEPTH, XGB_LEARNING_RATE, XGB_COLSAMPLE_BYTREE, XGB_N_ESTIMATORS
)

## 17.5. Predict

`load_xgb_model` restores the trees, the objective and the class count from the archive, but the
run-time settings come from the fresh `XGBClassifier()`, so `device` and `tree_method` have to be set
again — otherwise a booster trained on the card is reloaded onto the CPU without saying so.

In [ ]:
def load_xgb_model(model_name: str, subfolder: str, folder_name: str) -> XGBClassifier:
    """Rebuild the wrapper around a saved booster, back on the device it was trained on."""
    model = XGBClassifier()
    model.load_model(os.path.join(folder_name, subfolder, model_name))
    model.set_params(device=XGB_DEVICE, tree_method=XGB_TREE_METHOD)
    return model

In [ ]:
def xgb_predict(
    model_name: str,
    x,
    split_name: str,
    subfolder: str = XGB_SUBFOLDER,
    chunk_size: int = PREDICT_CHUNK_SIZE,
) -> np.ndarray:
    """Predict one split with a saved booster, caching each chunk."""
    with step(f"loading {model_name}"):
        model = load_xgb_model(model_name, subfolder, PATH_FOLDER_MODEL)

    predictions = predict_in_chunks(
        lambda rows: to_numpy(model.predict(to_xgb_features(rows))),
        x,
        get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name),
        "xgb",
        chunk_size,
    )

    del model
    free_gpu_memory()
    return predictions

In [ ]:
xgb_model_name = get_xgb_name(XGB_MAX_DEPTH, XGB_LEARNING_RATE, XGB_COLSAMPLE_BYTREE)
xgb_predict(xgb_model_name, dev_x, "dev")

In [ ]:
xgb_predict(xgb_model_name, test_x, "test")

## 17.6. Report

The last of the five. A booster of a few hundred shallow trees can be *cheaper* to predict 3.2M rows
with than a forest of 300 deep ones, so it is worth reading these numbers next to the
`predict_seconds` column of 17.3 — a model that matches the forest at a fraction of the prediction cost
is a result in its own right for a detector that has to keep up with live traffic.

In [ ]:
xgb_dev_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, XGB_SUBFOLDER), "dev")
xgb_test_folder = get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, XGB_SUBFOLDER), "test")

pd.concat(
    [
        get_evaluation_results(xgb_dev_folder, dev_y).assign(split="dev"),
        get_evaluation_results(xgb_test_folder, test_y).assign(split="test"),
    ],
    ignore_index=True,
)

In [ ]:
xgb_test_pred = load_prediction_chunks(xgb_test_folder)
get_classification_report(pred=xgb_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

In [ ]:
get_confusion_matrix(pred=xgb_test_pred, true=to_labels(test_y), class_names=CLASS_NAMES)

# 18. Results

The five models have each written their predictions for the same 3,246,594 test rows into
`prediction-result/<model>/test/`, so comparing them is a matter of scoring the same labels five times.
Nothing is refitted here and nothing new is decided — this section only reads what sections 13 to 17
produced.

Two tables and one figure, answering three different questions:

| | Question |
| --- | --- |
| 18.1 Overall | Which model is best, and does dev agree with test? |
| 18.2 Per class | Which attacks does each model actually catch? |
| 18.3 Cost | What does each model cost to train and to use? |

The second matters more than the first for an intrusion detector. A model that wins on macro F1 while
missing `SQL Injection` entirely is worse, operationally, than one a point behind that catches a few —
and the per-class view is the only place that shows up.

## 18.1. Overall

Every model that has cached predictions for a split is scored; a model whose section has not been run
is reported as missing rather than breaking the table.

Both splits are shown together on purpose. Dev is where the hyperparameters were chosen, so a model
that scores markedly better on dev than on test has been fitted to the validation split by the search
itself — with a 42-combination grid that is a real risk and worth seeing.

In [ ]:
MODEL_SUBFOLDERS = {
    "KNN": KNN_SUBFOLDER,
    "SVM": SVM_SUBFOLDER,
    "Random Forest": RF_SUBFOLDER,
    "Softmax": LOGREG_SUBFOLDER,
    "XGBoost": XGB_SUBFOLDER,
}

In [ ]:
def get_prediction_folder(subfolder: str, split_name: str) -> str:
    """`KNN` + `test` -> `prediction-result/KNN/test`."""
    return get_split_folder(os.path.join(PATH_FOLDER_PREDICTION, subfolder), split_name)

In [ ]:
def compare_models(
    subfolders: dict[str, str],
    true_labels_per_split: dict[str, object],
    score_column: str = SCORE_COLUMN,
) -> pd.DataFrame:
    """Score every model that has cached predictions, on every split given."""
    rows = []
    for model_name, subfolder in subfolders.items():
        for split_name, true_labels in true_labels_per_split.items():
            folder = get_prediction_folder(subfolder, split_name)
            try:
                scores = get_evaluation_results(folder, true_labels).iloc[0]
            except FileNotFoundError:
                print(f"{model_name} / {split_name}: no cached predictions, skipped.")
                continue
            rows.append({"model": model_name, "split": split_name, **scores.to_dict()})

    results = pd.DataFrame(rows)
    if results.empty:
        return results

    return results.sort_values(
        ["split", score_column], ascending=[True, False], ignore_index=True
    )

In [ ]:
model_comparison = compare_models(MODEL_SUBFOLDERS, {"dev": dev_y, "test": test_y})
model_comparison

## 18.2. Per Class

The macro F1 of 18.1 broken back into the fifteen numbers it averages, on the test split.

A column of this table is one model; a row is one class. Read it by row: a row that is dark across all
five models is an attack the features separate well, and a row that is light everywhere is one the
whole pipeline struggles with — which says more about the 24 components, the imputation, or SMOTE's
synthetic rows than about any classifier.

In [ ]:
def compare_per_class_f1(
    subfolders: dict[str, str],
    true_labels,
    split_name: str,
    class_names: list[str],
) -> pd.DataFrame:
    """Per-class F1 of every model on one split, as classes x models."""
    true = to_labels(true_labels)
    columns = {}

    for model_name, subfolder in subfolders.items():
        folder = get_prediction_folder(subfolder, split_name)
        try:
            pred = load_prediction_chunks(folder)
        except FileNotFoundError:
            print(f"{model_name}: no cached predictions for {split_name}, skipped.")
            continue

        report = get_classification_report(pred=pred, true=true, class_names=class_names)
        columns[model_name] = report.set_index("class")["f1"]

    return pd.DataFrame(columns).reindex(class_names)

In [ ]:
def plot_per_class_f1(per_class_f1: pd.DataFrame, split_name: str = "test") -> None:
    """Draw the per-class F1 of every model as a heatmap, one hue from light to dark."""
    surface, ink, muted = "#fcfcfb", "#0b0b0b", "#52514e"
    ramp = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
    colours = matplotlib.colors.LinearSegmentedColormap.from_list("f1", ramp)

    values = per_class_f1.to_numpy(dtype=float)
    figure, axes = plt.subplots(figsize=(1.6 * per_class_f1.shape[1] + 3, 0.42 * len(per_class_f1) + 2))
    figure.patch.set_facecolor(surface)

    axes.imshow(values, cmap=colours, vmin=0.0, vmax=1.0, aspect="auto")

    axes.set_xticks(range(per_class_f1.shape[1]))
    axes.set_xticklabels(per_class_f1.columns, fontsize=9)
    axes.set_yticks(range(len(per_class_f1)))
    axes.set_yticklabels(per_class_f1.index, fontsize=9)
    axes.set_title(f"Per-class F1 on the {split_name} split", color=ink, fontsize=11, loc="left", pad=12)
    axes.tick_params(colors=muted, length=0)
    for side in ("top", "right", "bottom", "left"):
        axes.spines[side].set_visible(False)

    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            if np.isnan(value):
                continue
            axes.text(
                column, row, f"{value:.2f}",
                ha="center", va="center", fontsize=9,
                color="#ffffff" if value > 0.55 else ink,
            )

    figure.tight_layout()
    plt.show()

In [ ]:
per_class_f1 = compare_per_class_f1(MODEL_SUBFOLDERS, test_y, "test", CLASS_NAMES)
per_class_f1.round(3)

In [ ]:
plot_per_class_f1(per_class_f1)

## 18.3. Cost

Accuracy is not the only axis a detector is judged on. The search tables already measured what each
model costs, so the last table collects the winning row of each of them: how long the chosen
combination took to fit, and how long it took to predict the search's dev subsample.

Two caveats belong with these numbers. They come from the *search*, so they were measured on subsamples
rather than on the full split, and on whatever else the machine was doing at the time — they are an
order-of-magnitude comparison, not a benchmark. And KNN's search does not fit per combination at all
(13.2), so it has no timing column to report.

In [ ]:
def compare_search_cost(
    search_results: dict[str, pd.DataFrame], score_column: str = SCORE_COLUMN
) -> pd.DataFrame:
    """The winning row of each search table, with whatever it recorded about its cost."""
    rows = []
    for model_name, results in search_results.items():
        if results is None or results.empty or score_column not in results:
            continue

        best = results.sort_values(score_column, ascending=False).iloc[0]
        rows.append(
            {
                "model": model_name,
                score_column: best[score_column],
                "fit_seconds": best.get("fit_seconds", float("nan")),
                "predict_seconds": best.get("predict_seconds", float("nan")),
                "n_support": best.get("n_support", float("nan")),
                "n_trees_used": best.get("n_trees_used", float("nan")),
                "epochs_run": best.get("epochs_run", float("nan")),
            }
        )

    return pd.DataFrame(rows).sort_values(score_column, ascending=False, ignore_index=True)

In [ ]:
compare_search_cost(
    {
        "KNN": knn_search_results,
        "SVM": svm_search_results,
        "Random Forest": rf_search_results,
        "Softmax": logreg_search_results,
        "XGBoost": xgb_search_results,
    }
)

Read the three tables together rather than separately.

The overall table says which model to report; the per-class heatmap says what that model would actually
do to traffic — and the classes that stay light in it are the ones to discuss honestly, particularly
`SQL Injection`, `Brute Force -XSS` and `Brute Force -Web`, whose training rows are almost entirely
SMOTE's interpolations (10.6). A high F1 on those three would say as much about synthetic neighbours as
about detection.

The cost table is what turns a result into a recommendation. If the booster and the forest are within a
point of each other and one of them predicts an order of magnitude faster, that is the one a detector
would run — and if the softmax baseline is within a point of both, the honest conclusion is that this
problem, in these 24 components, is close to linearly separable and the ensembles are buying very
little.